In [1]:
# ============================================================
# OUTPUT 4 / 40B03C1
# SoilGrids Core Texture Enrichment — SIAK_AOI01
# LOCAL PYTHON / JUPYTER VERSION
#
# PURPOSE
# - Read final exact 500 m ASU GeoJSON
# - Pull SoilGrids mean layers directly from official ISRIC WebDAV
# - Use only PALMSIM-core texture variables first:
#     sand, silt, clay
# - Depth-weight 0–100 cm:
#     0–5, 5–15, 15–30, 30–60, 60–100 cm
# - Calculate per-ASU mean, std, count
# - Export enriched GeoJSON + CSV + QC JSON
#
# NOTE
# - This avoids Google Earth Engine entirely.
# - Internet is required on the machine running this script.
# - SoilGrids native resolution is 250 m.
# ============================================================

from pathlib import Path
import json
import math
import sys
import subprocess
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. USER CONFIG
# ------------------------------------------------------------

# Put this script in the same folder as the GeoJSON,
# OR edit WORKDIR below.
WORKDIR = Path.cwd()

ASU_FILE = WORKDIR / "40B03A_FINAL_ASU_500M_EXACT_SIAK_AOI01_V1.geojson"

OUTPUT_GEOJSON = WORKDIR / "40B03C1_FINAL_ASU_500M_SOIL_TEXTURE_SIAK_AOI01_V1.geojson"
OUTPUT_CSV = WORKDIR / "40B03C1_FINAL_ASU_500M_SOIL_TEXTURE_SIAK_AOI01_V1.csv"
OUTPUT_QC = WORKDIR / "40B03C1_SOIL_TEXTURE_QC_SUMMARY_SIAK_AOI01_V1.json"

CACHE_DIR = WORKDIR / "soilgrids_cache_siak"
CACHE_DIR.mkdir(exist_ok=True)

TARGET_CRS = "EPSG:32647"
TARGET_RES_M = 250

# A small buffer ensures edge ASUs are fully covered.
BBOX_BUFFER_M = 500

DEPTHS = [
    ("0-5cm", 5),
    ("5-15cm", 10),
    ("15-30cm", 15),
    ("30-60cm", 30),
    ("60-100cm", 40),
]

PROPERTIES = {
    "sand": {"factor": 10.0, "unit": "%"},
    "silt": {"factor": 10.0, "unit": "%"},
    "clay": {"factor": 10.0, "unit": "%"},
}

# ------------------------------------------------------------
# 1. IMPORTS WITH FRIENDLY ERROR
# ------------------------------------------------------------

try:
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import rasterio
    from rasterio.mask import mask
    from osgeo import gdal
except Exception as e:
    print("\nERROR: Required geospatial packages are missing.")
    print("Recommended installation in Anaconda/Miniconda:")
    print("  conda install -c conda-forge geopandas rasterio gdal pandas numpy")
    print("\nThen restart Jupyter and run this script again.")
    raise

gdal.UseExceptions()

# ------------------------------------------------------------
# 2. LOAD FINAL ASU
# ------------------------------------------------------------

if not ASU_FILE.exists():
    raise FileNotFoundError(
        f"ASU file not found:\n{ASU_FILE}\n"
        "Place 40B03A_FINAL_ASU_500M_EXACT_SIAK_AOI01_V1.geojson "
        "in the same folder as this script or edit WORKDIR."
    )

asu = gpd.read_file(ASU_FILE)

if asu.crs is None:
    asu = asu.set_crs("EPSG:4326")

asu_utm = asu.to_crs(TARGET_CRS)

print("====================================================")
print("40B03C1 — SOILGRIDS CORE TEXTURE")
print("====================================================")
print("ASU count:", len(asu_utm))
print("ASU CRS:", asu_utm.crs)

xmin, ymin, xmax, ymax = asu_utm.total_bounds
xmin -= BBOX_BUFFER_M
ymin -= BBOX_BUFFER_M
xmax += BBOX_BUFFER_M
ymax += BBOX_BUFFER_M

print("Buffered UTM bbox:", (xmin, ymin, xmax, ymax))

# ------------------------------------------------------------
# 3. DOWNLOAD/CROP A SOILGRIDS LAYER USING OFFICIAL WEBDAV VRT
# ------------------------------------------------------------

def local_soilgrid_layer(prop, depth):
    """
    Use GDAL /vsicurl against official ISRIC SoilGrids WebDAV VRT.
    Only the SIAK subset is materialized locally.
    """
    out_tif = CACHE_DIR / f"{prop}_{depth}_mean_SIAK_UTM47N.tif"

    if out_tif.exists() and out_tif.stat().st_size > 0:
        return out_tif

    remote_vrt = (
        f"/vsicurl?max_retry=5&retry_delay=2&list_dir=no&"
        f"url=https://files.isric.org/soilgrids/latest/data/"
        f"{prop}/{prop}_{depth}_mean.vrt"
    )

    print(f"Fetching {prop} {depth} ...")

    options = gdal.WarpOptions(
        format="GTiff",
        dstSRS=TARGET_CRS,
        outputBounds=(xmin, ymin, xmax, ymax),
        outputBoundsSRS=TARGET_CRS,
        xRes=TARGET_RES_M,
        yRes=TARGET_RES_M,
        resampleAlg="bilinear",
        multithread=True,
        creationOptions=[
            "TILED=YES",
            "COMPRESS=DEFLATE",
            "PREDICTOR=2",
        ],
    )

    ds = gdal.Warp(
        str(out_tif),
        remote_vrt,
        options=options,
    )

    if ds is None:
        raise RuntimeError(f"GDAL failed for {prop} {depth}")

    ds = None

    if not out_tif.exists() or out_tif.stat().st_size == 0:
        raise RuntimeError(f"Output raster missing for {prop} {depth}")

    return out_tif

# ------------------------------------------------------------
# 4. CREATE DEPTH-WEIGHTED 0–100 CM RASTER
# ------------------------------------------------------------

def build_weighted_texture(prop, conversion_factor):
    """
    SoilGrids mapped integer values are converted to conventional units,
    then depth-weighted across 0–100 cm.
    """
    arrays = []
    weights = []
    profile = None

    for depth, thickness_cm in DEPTHS:
        tif = local_soilgrid_layer(prop, depth)

        with rasterio.open(tif) as src:
            arr = src.read(1).astype("float64")
            nodata = src.nodata
            if nodata is not None:
                arr[arr == nodata] = np.nan

            # Convert mapped units to conventional units.
            arr = arr / conversion_factor

            arrays.append(arr)
            weights.append(thickness_cm)

            if profile is None:
                profile = src.profile.copy()

    stack = np.stack(arrays, axis=0)
    weights_arr = np.asarray(weights, dtype="float64")[:, None, None]

    valid = np.isfinite(stack)
    weighted_sum = np.nansum(stack * weights_arr, axis=0)
    weight_sum = np.sum(valid * weights_arr, axis=0)

    out = np.full(stack.shape[1:], np.nan, dtype="float32")
    ok = weight_sum > 0
    out[ok] = (weighted_sum[ok] / weight_sum[ok]).astype("float32")

    out_path = CACHE_DIR / f"{prop}_0_100cm_weighted_SIAK_UTM47N.tif"

    profile.update(
        dtype="float32",
        count=1,
        nodata=-9999.0,
        compress="deflate",
        tiled=True,
    )

    write_arr = np.where(np.isfinite(out), out, -9999.0).astype("float32")

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(write_arr, 1)

    return out_path

# ------------------------------------------------------------
# 5. ZONAL STATS
# ------------------------------------------------------------

def zonal_stats_for_raster(gdf_utm, raster_path, prefix):
    means = []
    stds = []
    counts = []

    with rasterio.open(raster_path) as src:
        nodata = src.nodata

        for geom in gdf_utm.geometry:
            try:
                data, _ = mask(
                    src,
                    [geom],
                    crop=True,
                    all_touched=True,
                    filled=False,
                )
                vals = data[0]

                if hasattr(vals, "compressed"):
                    vals = vals.compressed()
                else:
                    vals = vals[np.isfinite(vals)]

                vals = np.asarray(vals, dtype="float64")

                if nodata is not None:
                    vals = vals[vals != nodata]

                vals = vals[np.isfinite(vals)]

                if vals.size == 0:
                    means.append(np.nan)
                    stds.append(np.nan)
                    counts.append(0)
                else:
                    means.append(float(np.mean(vals)))
                    stds.append(float(np.std(vals, ddof=0)))
                    counts.append(int(vals.size))

            except Exception:
                means.append(np.nan)
                stds.append(np.nan)
                counts.append(0)

    gdf_utm[f"{prefix}_MEAN"] = means
    gdf_utm[f"{prefix}_STD"] = stds
    gdf_utm[f"{prefix}_COUNT"] = counts

    return gdf_utm

# ------------------------------------------------------------
# 6. RUN
# ------------------------------------------------------------

weighted_files = {}

for prop, cfg in PROPERTIES.items():
    weighted_files[prop] = build_weighted_texture(
        prop,
        cfg["factor"]
    )

for prop in ["sand", "silt", "clay"]:
    prefix = prop.upper() + "_PCT_0_100"
    asu_utm = zonal_stats_for_raster(
        asu_utm,
        weighted_files[prop],
        prefix,
    )

# ------------------------------------------------------------
# 7. BASIC TEXTURE QC
# ------------------------------------------------------------

asu_utm["TEXTURE_SUM_PCT"] = (
    asu_utm["SAND_PCT_0_100_MEAN"]
    + asu_utm["SILT_PCT_0_100_MEAN"]
    + asu_utm["CLAY_PCT_0_100_MEAN"]
)

asu_utm["SOIL_TEXTURE_DATA_VALID"] = (
    asu_utm[
        [
            "SAND_PCT_0_100_MEAN",
            "SILT_PCT_0_100_MEAN",
            "CLAY_PCT_0_100_MEAN",
        ]
    ]
    .notna()
    .all(axis=1)
    .astype(int)
)

# SoilGrids values should approximately sum to 100%.
asu_utm["TEXTURE_SUM_QC"] = np.where(
    asu_utm["SOIL_TEXTURE_DATA_VALID"].eq(0),
    "NO_DATA",
    np.where(
        asu_utm["TEXTURE_SUM_PCT"].between(95, 105),
        "PASS",
        "CHECK",
    ),
)

asu_utm["SOIL_SOURCE"] = "SoilGrids_2.0_mean_250m_ISRIC_WebDAV"
asu_utm["SOIL_OBSERVED_LOCAL"] = 0
asu_utm["SOIL_DEPTH_PROFILE_USED"] = "0-100cm_depth_weighted"
asu_utm["SOIL_ENRICHMENT_VERSION"] = "40B03C1_V1"

# ------------------------------------------------------------
# 8. EXPORT
# ------------------------------------------------------------

asu_out = asu_utm.to_crs("EPSG:4326")
asu_out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")

csv_df = pd.DataFrame(asu_out.drop(columns="geometry"))
csv_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

valid = asu_utm["SOIL_TEXTURE_DATA_VALID"].eq(1)

qc = {
    "status": "PASS" if valid.all() else "PASS_WITH_MISSING",
    "asu_count": int(len(asu_utm)),
    "soil_texture_valid_asu_count": int(valid.sum()),
    "soil_texture_missing_asu_count": int((~valid).sum()),
    "soil_texture_valid_fraction": float(valid.mean()),
    "texture_sum_pass_count": int((asu_utm["TEXTURE_SUM_QC"] == "PASS").sum()),
    "texture_sum_check_count": int((asu_utm["TEXTURE_SUM_QC"] == "CHECK").sum()),
    "sand_mean_pct_global_asu_mean": float(asu_utm.loc[valid, "SAND_PCT_0_100_MEAN"].mean()),
    "silt_mean_pct_global_asu_mean": float(asu_utm.loc[valid, "SILT_PCT_0_100_MEAN"].mean()),
    "clay_mean_pct_global_asu_mean": float(asu_utm.loc[valid, "CLAY_PCT_0_100_MEAN"].mean()),
    "source": "ISRIC SoilGrids 2.0 mean predictions via official WebDAV VRT",
    "resolution_m": 250,
    "depths_used_cm": ["0-5", "5-15", "15-30", "30-60", "60-100"],
    "output_geojson": OUTPUT_GEOJSON.name,
    "output_csv": OUTPUT_CSV.name,
}

with open(OUTPUT_QC, "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2)

print("====================================================")
print("40B03C1 FINISHED")
print("====================================================")
print("ASU count:", qc["asu_count"])
print("Valid soil-texture ASUs:", qc["soil_texture_valid_asu_count"])
print("Missing soil-texture ASUs:", qc["soil_texture_missing_asu_count"])
print("Texture-sum PASS:", qc["texture_sum_pass_count"])
print("Texture-sum CHECK:", qc["texture_sum_check_count"])
print("Output GeoJSON:", OUTPUT_GEOJSON)
print("Output CSV:", OUTPUT_CSV)
print("QC JSON:", OUTPUT_QC)
print("====================================================")


ERROR: Required geospatial packages are missing.
Recommended installation in Anaconda/Miniconda:
  conda install -c conda-forge geopandas rasterio gdal pandas numpy

Then restart Jupyter and run this script again.


ModuleNotFoundError: No module named 'rasterio'

In [2]:
import sys
print(sys.executable)

D:\New folder\envs\copernicus_env\python.exe


In [1]:
import rasterio
print(rasterio.__version__)

1.4.3


In [2]:
from osgeo import gdal
print(gdal.VersionInfo())

ModuleNotFoundError: No module named 'osgeo'

In [1]:
from osgeo import gdal
print(gdal.VersionInfo())

3100300


In [2]:
import os
print(os.getcwd())

C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN


In [3]:
import os

print(os.path.exists("40B03A_FINAL_ASU_500M_EXACT_SIAK_AOI01_V1.geojson"))
print(os.path.exists("40B03C1_SOILGRIDS_TEXTURE_LOCAL_PYTHON_SIAK_AOI01.py"))

False
False


In [4]:
from pathlib import Path

WORKDIR = Path(r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD")

print((WORKDIR / "40B03A_FINAL_ASU_500M_EXACT_SIAK_AOI01_V1.geojson").exists())
print((WORKDIR / "40B03C1_SOILGRIDS_TEXTURE_LOCAL_PYTHON_SIAK_AOI01.py").exists())

True
True


In [5]:
import os

os.chdir(r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD")

print(os.getcwd())

C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD


In [6]:
from pathlib import Path
import os
import sys

# ============================================================
# 40B03C1 — SETUP + PRECHECK + RUN
# ============================================================

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

os.chdir(WORKDIR)

asu_file = WORKDIR / "40B03A_FINAL_ASU_500M_EXACT_SIAK_AOI01_V1.geojson"
script_file = WORKDIR / "40B03C1_SOILGRIDS_TEXTURE_LOCAL_PYTHON_SIAK_AOI01.py"

print("Working directory :", os.getcwd())
print("Python            :", sys.executable)
print("ASU GeoJSON       :", asu_file.exists())
print("40B03C1 script    :", script_file.exists())

assert asu_file.exists(), "ERROR: Final ASU GeoJSON tidak ditemukan."
assert script_file.exists(), "ERROR: Script 40B03C1 tidak ditemukan."

# Check required packages
import numpy
import pandas
import geopandas
import rasterio
from osgeo import gdal

print("\nPackage check: PASS")
print("Rasterio :", rasterio.__version__)
print("GDAL     :", gdal.VersionInfo())

# Run SoilGrids enrichment
print("\n====================================================")
print("STARTING 40B03C1 SOILGRIDS ENRICHMENT")
print("====================================================\n")

exec(
    compile(
        script_file.read_text(encoding="utf-8"),
        str(script_file),
        "exec"
    )
)

Working directory : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD
Python            : D:\New folder\envs\copernicus_env\python.exe
ASU GeoJSON       : True
40B03C1 script    : True

Package check: PASS
Rasterio : 1.4.3
GDAL     : 3100300

STARTING 40B03C1 SOILGRIDS ENRICHMENT

40B03C1 — SOILGRIDS CORE TEXTURE
ASU count: 820
ASU CRS: EPSG:32647
Buffered UTM bbox: (np.float64(804065.1000015538), np.float64(71745.10009765995), np.float64(820084.8999046646), np.float64(87764.71875012449))
Fetching sand 0-5cm ...
Fetching sand 5-15cm ...
Fetching sand 15-30cm ...
Fetching sand 30-60cm ...
Fetching sand 60-100cm ...
Fetching silt 0-5cm ...
Fetching silt 5-15cm ...
Fetching silt 15-30cm ...
Fetching silt 30-60cm ...
Fetching silt 60-100cm ...
Fetching clay 0-5cm ...
Fetching clay 5-15cm ...
Fetching clay 15-30cm ...
Fetching clay 30-60cm ...
Fetching clay 60-100cm ...
40B03C1 FINISHED
ASU count: 820
Valid soil-texture ASUs: 820
Missing soil-texture ASUs: 0
Texture-sum PAS

In [7]:
%run 40B04A_CLIMATE_NATIVE_CELL_MAPPING_SIAK_AOI01_V1.py

40B04A — CLIMATE NATIVE-CELL MAPPING
ASU count: 820
Centroid bbox: 101.737890426614 0.6533373813435813 101.87072842963626 0.787811349726836

Reading CHIRPS native grid metadata...
CHIRPS CRS: EPSG:4326
CHIRPS resolution: (0.05000000074505806, 0.05000000074505806)
CHIRPS raster size: 7200 x 2000

40B04A FINISHED
ASU mapped to CHIRPS: 820 / 820
ASU mapped to ERA5-Land: 820 / 820
Unique CHIRPS cells: 12
Unique ERA5-Land cells: 6
Output mapping: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04A_ASU_CLIMATE_CELL_MAPPING_SIAK_AOI01_V1.csv
Unique CHIRPS: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04A_UNIQUE_CHIRPS_CELLS_SIAK_AOI01_V1.csv
Unique ERA5-Land: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04A_UNIQUE_ERA5L_CELLS_SIAK_AOI01_V1.csv
QC: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04A_CLIMATE_CELL_MAPPING_QC_SIAK_AOI01_V1.json


In [8]:
%run 40B04B1_CHIRPS_V2_DAILY_FORCING_SIAK_AOI01_V1.py

40B04B1 — CHIRPS v2 DAILY RAINFALL
Unique CHIRPS cells: 12
Period: 2001-01-01 → 2025-12-31
Expected days/cell: 9131
Expected final rows: 109572
Checkpoint folder: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\climate_raw\chirps_v2

----------------------------------------------------
Cell 1/12: CHIRPS_R984_C5634 @ (101.725004, 0.774999)
----------------------------------------------------
  request 2001-01-01 → 2010-12-31
    job: 4e0c86ec-3687-4b07-91f0-1f6ba894ccb5
    progress ~8.5%
    progress ~31.2%
    progress ~56.7%
    progress ~82.2%
    progress ~100.0%
  request 2011-01-01 → 2020-12-31
    job: 35e6b1dd-52e6-4e9d-91c1-c8d995a861a4
    progress ~5.7%
    progress ~34.0%
    progress ~59.5%
    progress ~85.0%
    progress ~100.0%
  request 2021-01-01 → 2025-12-31
    job: 24d34088-4a8c-4403-a099-af54968963a6
    progress ~6.2%
    progress ~74.0%
    progress ~100.0%

----------------------------------------------------
Cell 2/12: CHIRPS_R984_C5635 @ (1

In [9]:
from pathlib import Path
import importlib.util

print("cdsapi installed :", importlib.util.find_spec("cdsapi") is not None)
print("CDS credential    :", (Path.home() / ".cdsapirc").exists())
print("Credential path   :", Path.home() / ".cdsapirc")

cdsapi installed : True
CDS credential    : True
Credential path   : C:\Users\user\.cdsapirc


In [10]:
%run 40B04B2_ERA5LAND_DAILY_FORCING_ARCO_SIAK_AOI01_V1.py


ERROR: package untuk akses ERA5-Land ARCO belum lengkap.
Missing: aiohttp

Di Anaconda Prompt dengan copernicus_env aktif jalankan:
conda install -c conda-forge xarray zarr fsspec aiohttp

Setelah selesai: restart Jupyter kernel lalu run script ini lagi.


SystemExit: 1

In [1]:
import aiohttp
print(aiohttp.__version__)

SSLError: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4040)

In [2]:
import ssl
import certifi

_original_create_default_context = ssl.create_default_context

def create_default_context_certifi(
    purpose=ssl.Purpose.SERVER_AUTH,
    cafile=None,
    capath=None,
    cadata=None
):
    if cafile is None and capath is None and cadata is None:
        cafile = certifi.where()

    return _original_create_default_context(
        purpose=purpose,
        cafile=cafile,
        capath=capath,
        cadata=cadata
    )

ssl.create_default_context = create_default_context_certifi

import aiohttp

print("aiohttp version :", aiohttp.__version__)
print("CA bundle       :", certifi.where())

# Restore Python function after aiohttp has created its verified SSL contexts.
ssl.create_default_context = _original_create_default_context

aiohttp version : 3.14.3
CA bundle       : D:\New folder\envs\copernicus_env\lib\site-packages\certifi\cacert.pem


In [3]:
%run 40B04B2_ERA5LAND_DAILY_FORCING_ARCO_SIAK_AOI01_V1.py

Exception: File `'40B04B2_ERA5LAND_DAILY_FORCING_ARCO_SIAK_AOI01_V1.py'` not found.

In [4]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_DAILY_FORCING_ARCO_SIAK_AOI01_V1.py"

40B04B2 — ERA5-LAND DAILY METEOROLOGICAL FORCING
Python: D:\New folder\envs\copernicus_env\python.exe
xarray: 2025.6.1
Period: 2001-01-01T00:00:00 → 2025-12-31T23:00:00
CDS credential: FOUND
Credential path: C:\Users\user\.cdsapirc
Unique ERA5-Land cells: 6

----------------------------------------------------
Opening: temperature
----------------------------------------------------
  T2M_K <- t2m
  TD2M_K <- d2m
  loading selected 6-point time-series...
  checkpoint: 40B04B2_temperature_6cells_2001_2025.nc
  rows: 1314864

----------------------------------------------------
Opening: pressure_precipitation
----------------------------------------------------
  SP_PA <- sp
  TP_M_HOURLY <- tp
  loading selected 6-point time-series...
  checkpoint: 40B04B2_pressure_precipitation_6cells_2001_2025.nc
  rows: 1314864

----------------------------------------------------
Opening: wind
----------------------------------------------------
  U10_MS <- u10
  V10_MS <- v10
  loading selected 6-p

MemoryError: 

In [5]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_DAILY_FORCING_MEMORY_SAFE_RESUME_SIAK_AOI01_V2.py"

40B04B2 V2 — MEMORY-SAFE RESUME
Python: D:\New folder\envs\copernicus_env\python.exe
xarray: 2025.6.1
CA bundle: D:\New folder\envs\copernicus_env\lib\site-packages\certifi\cacert.pem
Expected daily rows: 54786
Unique ERA5-Land cells: 6

Checkpoint audit:
  temperature             : FOUND | 40B04B2_temperature_6cells_2001_2025.nc
  pressure_precipitation  : FOUND | 40B04B2_pressure_precipitation_6cells_2001_2025.nc
  wind                    : FOUND | 40B04B2_wind_6cells_2001_2025.nc

[1/4] Reusing temperature checkpoint...
  daily rows: 54786

[2/4] Reusing pressure/precipitation checkpoint...
  daily rows: 54786

[3/4] Reusing wind checkpoint...
  daily rows: 54786

[4/4] Retrieving radiation only — memory-safe mode...
  Strategy: chunks=None + quarterly cache
  SSRD variable: ssrd
  fetching 2001 Q1 ... 

MemoryError: 

In [6]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_CDS_SERVER_SIDE_RADIATION_SIAK_AOI01_V3.py"

40B04B2 V3 — CDS SERVER-SIDE RADIATION RETRIEVAL
Python: D:\New folder\envs\copernicus_env\python.exe
Unique ERA5-Land cells: 6
Period: 2001-01-01 → 2025-12-31

Checkpoint audit:
  temperature             : FOUND | 40B04B2_temperature_6cells_2001_2025.nc
  pressure_precipitation  : FOUND | 40B04B2_pressure_precipitation_6cells_2001_2025.nc
  wind                    : FOUND | 40B04B2_wind_6cells_2001_2025.nc

[1/4] Reusing temperature checkpoint...
  daily rows: 54786

[2/4] Reusing pressure/precipitation checkpoint...
  daily rows: 54786

[3/4] Reusing wind checkpoint...
  daily rows: 54786

[4/4] Radiation via official CDS time-series service...
  CDS area [N,W,S,E]: [0.81, 101.69, 0.69, 101.91000000000001]
  requesting server-side NetCDF...
  NOTE: CDS time-series service may return the full available time range;
        V3 filters locally to 2001-2025 after download.


2026-08-18 15:21:31,239 INFO Request ID is b9c625af-511a-43a0-a190-dd6e72ea59f9
2026-08-18 15:21:31,572 INFO status has been updated to accepted
2026-08-18 15:21:54,595 INFO status has been updated to running
2026-08-18 15:22:06,376 INFO status has been updated to failed


HTTPError: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/b9c625af-511a-43a0-a190-dd6e72ea59f9/results
The job has failed
The job failed with: MultiAdaptorNoDataError

In [1]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_DAILY_FORCING_V4_STANDARD_CDS_SSRD_SIAK_AOI01.py"

40B04B2 V4 — STANDARD CDS DAILY SSRD FALLBACK
Python: D:\New folder\envs\copernicus_env\python.exe
Unique ERA5-Land cells: 6
Period: 2001-01-01 → 2025-12-31
Expected final daily rows: 54786
CDS area [N,W,S,E]: [0.81, 101.69, 0.69, 101.91000000000001]

Checkpoint audit:
  temperature             : FOUND | 40B04B2_temperature_6cells_2001_2025.nc
  pressure_precipitation  : FOUND | 40B04B2_pressure_precipitation_6cells_2001_2025.nc
  wind                    : FOUND | 40B04B2_wind_6cells_2001_2025.nc

[1/4] Reusing temperature checkpoint...
  daily rows: 54786

[2/4] Reusing pressure/precipitation checkpoint...
  daily rows: 54786

[3/4] Reusing wind checkpoint...
  daily rows: 54786

[4/4] Retrieving daily SSRD from standard ERA5-Land CDS...
  Method: 00 UTC accumulated SSRD -> shift date -1 day
  This avoids all direct ARCO/Zarr radiation access.
  requesting 2001-2005 ...


2026-08-19 09:13:18,125 INFO Request ID is 3c38c783-8593-4a79-bae4-56e568f50cf8
2026-08-19 09:13:18,349 INFO status has been updated to accepted
2026-08-19 09:13:32,973 INFO status has been updated to running
2026-08-19 09:19:42,286 INFO status has been updated to successful


d3a87b246a18ce9707c21770504c15d0.nc:   0%|          | 0.00/166k [00:00<?, ?B/s]

    downloaded: SSRD_00UTC_2001_2005_SIAK.nc
  requesting 2006-2010 ...


2026-08-19 09:19:48,701 INFO Request ID is 6ad69e31-2199-4294-a2a1-b303b0d753de
2026-08-19 09:19:48,987 INFO status has been updated to accepted
2026-08-19 09:20:00,569 INFO status has been updated to running
2026-08-19 09:26:13,718 INFO status has been updated to successful


4c93bcc14518e48e23022582a12fff9b.nc:   0%|          | 0.00/166k [00:00<?, ?B/s]

... (9 outputs left out)

    downloaded: SSRD_00UTC_2021_2025_SIAK.nc
  requesting 2026-01-01 tail timestamp...


2026-08-19 09:45:49,666 INFO Request ID is 7a9f4c75-2257-4313-afc9-602388ef18d9
2026-08-19 09:45:49,868 INFO status has been updated to accepted
2026-08-19 09:46:04,314 INFO status has been updated to running
2026-08-19 09:46:12,198 INFO status has been updated to successful


5f87c49ca2ce8eb22748c7f329417c62.nc:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

    downloaded: SSRD_00UTC_2026_01_01_SIAK.nc
  radiation daily rows: 54786 / 54786

Merging daily forcing tables...
  merged daily rows: 54786 / 54786

40B04B2 V4 FINISHED
Status: PASS
Daily rows: 54786 / 54786
Complete cells: 6 / 6
Missing forcing days: 0
Plausibility: PASS
Daily output: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_DAILY_NATIVE_CELLS_2001_2025_SIAK_AOI01_V4.csv
Cell QC: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_CELL_QC_SIAK_AOI01_V4.csv
Monthly: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_MONTHLY_NATIVE_CELLS_2001_2025_SIAK_AOI01_V4.csv
Annual: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_ANNUAL_NATIVE_CELLS_2001_2025_SIAK_AOI01_V4.csv
QC JSON: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04B2_ERA5LAND_QC_SUMMARY_SIAK_AOI01_V4.json


In [ ]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04C_FINAL_CLIMATE_HARMONIZATION_SIAK_AOI01_V1.py"

40B04C — FINAL CLIMATE HARMONIZATION
ASU rows: 820
CHIRPS daily rows: 109572
ERA5-Land daily rows: 54786
Unique climate pairs: 12


In [ ]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40B04C_FINAL_CLIMATE_HARMONIZATION_SIAK_AOI01_V1.py"

40B04C — FINAL CLIMATE HARMONIZATION
ASU rows: 820
CHIRPS daily rows: 109572
ERA5-Land daily rows: 54786
Unique climate pairs: 12


In [ ]:
%run "C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD_Producifity\40B04C_FINAL_CLIMATE_HARMONIZATION_SIAK_AOI01_V1.py"

40B04C — FINAL CLIMATE HARMONIZATION
ASU rows: 820
CHIRPS daily rows: 109572
ERA5-Land daily rows: 54786
Unique climate pairs: 12


In [1]:
import os
import sys
import subprocess
import gc

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

os.chdir(WORKDIR)

print("Python :", sys.executable)
print("Folder :", os.getcwd())

gc.collect()

Python : D:\New folder\envs\copernicus_env\python.exe
Folder : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD


8

In [2]:
subprocess.run(
    [
        sys.executable,
        "-u",
        "NAMA_SCRIPT.py"
    ],
    cwd=WORKDIR
)

CompletedProcess(args=['D:\\New folder\\envs\\copernicus_env\\python.exe', '-u', 'NAMA_SCRIPT.py'], returncode=2)

In [3]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40B05_FINAL_MODEL_INPUT_PACKAGING_SIAK_AOI01_V2_JUPYTER_SAFE.py"
    ],
    cwd=r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

print("Return code:", result.returncode)

Return code: 0


In [4]:
import json
from pathlib import Path

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

qc_file = WORKDIR / "40B05_FINAL_MODEL_INPUT_PACKAGING_QC_SIAK_AOI01_V1.json"

print("QC file exists:", qc_file.exists())

if qc_file.exists():
    with open(qc_file, "r", encoding="utf-8") as f:
        qc = json.load(f)

    print("\n===== 40B05 RESULT =====")
    print("Status                  :", qc["status"])
    print("ASU count               :", qc["asu_count"])
    print("Age available           :", qc["coverage"]["age_available_asus"])
    print("Condition available     :", qc["coverage"]["condition_available_asus"])
    print("Soil texture complete   :", qc["coverage"]["soil_texture_complete_asus"])
    print("Climate refs complete   :", qc["coverage"]["climate_reference_complete_asus"])
    print("Latitude available      :", qc["coverage"]["latitude_available_asus"])
    print("PALMSIM eligible ASUs   :", qc["model_readiness"]["palmsim_static_core_eligible_asus"])
    print("APSIM weather-ready ASUs:", qc["model_readiness"]["apsim_weather_reference_ready_asus"])
    print("Next gate               :", qc["next_gate"])

QC file exists: True

===== 40B05 RESULT =====
Status                  : PASS
ASU count               : 820
Age available           : 654
Condition available     : 818
Soil texture complete   : 820
Climate refs complete   : 820
Latitude available      : 820
PALMSIM eligible ASUs   : 654
APSIM weather-ready ASUs: 820
Next gate               : 40C01 PALMSIM experiment adapter + explicit density/soil-depth sensitivity scenarios


In [5]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C01_PALMSIM_EXPERIMENT_PREFLIGHT_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C01 — PALMSIM EXPERIMENT PREFLIGHT
Python: D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

Frozen input:
  ASUs total          : 820
  PALMSIM eligible    : 654
  Held due core input : 166

Climate preflight:
  Required CHIRPS cells: 12
  Required ERA5L cells : 6
  Climate complete     : True

PALMSIM source discovery:
  Importable       : False
  Module origin    : None
  Reported version : None
  Candidate paths  : 0

Scenario design:
  Scenarios: 5
  Design   : one-at-a-time baseline sensitivity

Planned PALMSIM experiment:
  Eligible ASUs : 654
  Scenarios     : 5
  Planned runs  : 3270

40C01 FINISHED
Status: PASS_INPUTS_CHECK_MODEL_SOURCE
Eligible ASUs: 654 / 820
Climate complete: True
PALMSIM source discovered: False
Scenarios: 5
Planned runs: 3270
Scenario matrix: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40C01_PALMSIM_SENSITIVITY_SCENARIOS_SIAK_AOI01_V1.csv
Eligible ASUs: C:\Us

In [6]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C01B_REGISTER_PALMSIM_2_5_2_SOURCE_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C01B — PALMSIM 2.5.2 SOURCE REGISTER
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

Archive found: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\palmsim_master_0.zip
Archive SHA256: 3904b39bed0be00d570742acf6045ae332d49dc9eaa603c92fb7808a9787e4e9

Extracting immutable original copy...
  extracted: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_ORIGINAL
Working copy: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_WORKING

Compatibility patch:
  files patched : 1
  replacements  : 5

Source structure:
  Python files : 18
  Notebooks    : 9
  Package dirs : 3
  Version hits : 1

40C01B FINISHED
Status: PASS
Original source: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_ORIGINAL
Working source : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_WORKING


In [7]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C02A_PALMSIM_API_CONTRACT_AUDIT_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C02A — PALMSIM API CONTRACT / EXAMPLE DISCOVERY
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD
Working source: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_WORKING

Inventory:
  Python files : 18
  Notebooks    : 9
  CSV files    : 8
  TXT files    : 0

Contract discovery:
  Model-like classes : 2
  Run-like functions : 0
  Best ref notebook  : palmsim-master\examples\06 El Nino.ipynb

Top candidate notebooks:
  score= 41 | palmsim-master\examples\06 El Nino.ipynb
  score= 40 | palmsim-master\examples\01 Climate Change.ipynb
  score= 39 | palmsim-master\examples\08 Sensitivity analysis.ipynb
  score= 26 | palmsim-master\examples\02 Potential Production.ipynb
  score= 21 | palmsim-master\examples\07 Biomass Production.ipynb
  score= 19 | palmsim-master\examples\05 Vegetative Growth.ipynb
  score=  7 | palmsim-master\examples\04 Generative Growth.ipynb
  score=  5 | pa

In [8]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C02B_PALMSIM_REFERENCE_SMOKE_TEST_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C02B — PALMSIM OFFICIAL REFERENCE SMOKE TEST
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

Reference notebook:
  C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_WORKING\palmsim-master\examples\02 Potential Production.ipynb
Jupyter executable: D:\New folder\Scripts\jupyter.EXE

Executing official reference run 1/2 ...
  return code: 0
  stderr tail:
[NbConvertApp] Converting notebook C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_WORKING\palmsim-master\examples\02 Potential Production.ipynb to notebook
D:\New folder\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_lo

In [9]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C02C_PALMSIM_SIAK_1ASU_SMOKE_TEST_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C02C — PALMSIM SIAK 1-ASU SMOKE TEST
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD
PALMSIM master: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\PALMSIM_2.5.2_WORKING\palmsim-master
PalmField import: PASS

Selected ASU:
  UNIT_ID                 : SIAK_AOI01_G500_1633,144
  Age proxy (yr)          : 10.0
  Model age rounded (yr)  : 10
  Year-of-planting proxy  : 2015
  Simulation period       : 2015-01-01 -> 2024-12-28
  Duration                : 3650 days
  CHIRPS cell             : CHIRPS_R986_C5636
  ERA5-Land cell          : ERA5L_+00.7_+101.8
  Latitude                : 0.653991593224034

Weather adapter:
  Rows                    : 3650
  Missing required values : 0
  Complete calendar       : True

Running Siak PALMSIM smoke test 1/2 ...
  rows   : 365
  columns: 181

Running Siak PALMSIM smoke test 2/2 ...
  rows   : 365
  columns: 181
  latitude target(s): ['PalmField.lat

In [10]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C02D_PALMSIM_STRUCTURAL_PARAMETER_CONTRACT_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C02D — PALMSIM STRUCTURAL PARAMETER CONTRACT
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

PalmField instantiated: PASS
Year-of-planting proxy: 2015
ASU latitude          : 0.653991593224034

Confirmed operational contract:
  CONFIRMED: PalmField.year_of_planting
  CONFIRMED: PalmField.latitude
  CONFIRMED: PalmField.weather.radiation_series
  CONFIRMED: PalmField.weather.rainfall_series
  CONFIRMED: PalmField.run(duration=...)

Structural parameter discovery:

  [density]
   - PalmField.planting_density

  [soil_depth]
   - PalmField.soil.soil_depth

  [sand]
   - NONE FOUND

  [silt]
   - NONE FOUND

  [clay]
   - NONE FOUND

  [texture]
   - PalmField.soil._soil_texture_class

  [water]
   - PalmField.soil._plant_available_water_content
   - PalmField.soil.available_water
   - PalmField.female._stress_index_

Gate checks:
  density path found     : True
  soil-depth path found  : True
  sand path 

In [11]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C02E_PALMSIM_SOIL_DENSITY_SETTER_AUDIT_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C02E — PALMSIM SOIL / DENSITY SETTER AUDIT
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

PalmField class: palmsim.palm PalmField
Soil class     : palmsim.components.soil Soil

Exact live paths:
  PalmField.planting_density | instance_attribute | writable=True | value=143
  PalmField.soil.soil_depth | instance_attribute | writable=True | value=1
  PalmField.soil._soil_texture_class | instance_attribute | writable=True | value='loamy sand'
  PalmField.soil._plant_available_water_content | instance_attribute | writable=True | value=0.08605289880279436
  PalmField.soil.available_water | instance_attribute | writable=True | value=86.05289880279436

Texture-class candidates from source:
  - NONE

Gate checks:
  planting density found       : True
  soil depth found             : True
  public texture property      : False
  private texture attr         : True
  explicit texture candidates  : 0

40C02E FINI

In [12]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "40C03_PALMSIM_1ASU_P00_P04_SENSITIVITY_SIAK_AOI01_V1.py"
    ],
    cwd=WORKDIR,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\n===== ERROR / WARNING =====")
    print(result.stderr)

print("\nReturn code:", result.returncode)

40C03 — PALMSIM P00-P04 SINGLE-ASU SENSITIVITY
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD
PALMSIM import: PASS

Frozen smoke-test ASU:
  UNIT_ID                : SIAK_AOI01_G500_1633,144
  Year-of-planting proxy : 2015
  Duration days          : 3650
  Latitude               : 0.653991593224034
  Weather rows           : 3650

Scenario matrix:
      SCENARIO_ID  PALM_DENSITY_PALMS_HA  SOIL_DEPTH_M
         P00_BASE                  143.0           1.0
     P01_DENS_LOW                  130.0           1.0
    P02_DENS_HIGH                  150.0           1.0
P03_DEPTH_SHALLOW                  143.0           0.5
   P04_DEPTH_DEEP                  143.0           1.5

Running P00_BASE: density=143.0, depth=1.0 m
  repeat 1: rows=365, hash=35446cda87ee8e52...
  repeat 2: rows=365, hash=35446cda87ee8e52...

Running P01_DENS_LOW: density=130.0, depth=1.0 m
  repeat 1: rows=365, hash=e94b4ce7b9aa9fc8...


In [ ]:
import subprocess
import sys

WORKDIR = r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"

proc = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "40C04A_PALMSIM_P00_BASELINE_BATCH_SIAK_AOI01_V3_ROBUST_2WORKER.py"
    ],
    cwd=WORKDIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end="")

return_code = proc.wait()
print("\nReturn code:", return_code)

In [1]:
import os
import subprocess
from pathlib import Path
from datetime import datetime

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

print("===== JUPYTER CHECK =====")
print("Kernel Jupyter aktif ✅")
print("Kernel PID:", os.getpid())

# 1. Cek subprocess lama kalau variabel proc masih ada
print("\n===== SUBPROCESS CHECK =====")
if "proc" in globals():
    status = proc.poll()
    if status is None:
        print("proc masih RUNNING ✅")
        print("PID:", proc.pid)
    else:
        print("proc sudah STOP.")
        print("Return code:", status)
else:
    print("Variabel 'proc' tidak ada di kernel saat ini.")

# 2. Cari semua proses Python yang masih menjalankan script 40C04A
print("\n===== WINDOWS PROCESS CHECK =====")

ps_command = r'''
Get-CimInstance Win32_Process |
Where-Object {
    $_.CommandLine -like "*40C04A*" -or
    $_.CommandLine -like "*PALMSIM_P00_BASELINE*"
} |
Select-Object ProcessId, ParentProcessId, Name, CommandLine |
Format-List
'''

result = subprocess.run(
    ["powershell", "-NoProfile", "-Command", ps_command],
    capture_output=True,
    text=True
)

if result.stdout.strip():
    print(result.stdout)
else:
    print("Tidak ada proses 40C04A/PALMSIM baseline yang sedang aktif.")

# 3. Cek apakah file checkpoint benar-benar bertambah
print("\n===== CHECKPOINT CHECK =====")

files = [
    WORKDIR / "40C04A_PALMSIM_P00_BASELINE_CHECKPOINT_SIAK_AOI01_V1.csv",
    WORKDIR / "40C04A_PALMSIM_P00_BASELINE_RESULTS_SIAK_AOI01_V3.csv",
]

for f in files:
    if f.exists():
        try:
            with open(f, "r", encoding="utf-8-sig") as x:
                rows = max(sum(1 for _ in x) - 1, 0)

            modified = datetime.fromtimestamp(
                f.stat().st_mtime
            ).strftime("%Y-%m-%d %H:%M:%S")

            print(f"\n{f.name}")
            print("Rows         :", rows)
            print("Last modified:", modified)
        except Exception as e:
            print(f.name, "->", e)
    else:
        print(f"\n{f.name}")
        print("Belum ada.")

===== JUPYTER CHECK =====
Kernel Jupyter aktif ✅
Kernel PID: 6736

===== SUBPROCESS CHECK =====
Variabel 'proc' tidak ada di kernel saat ini.

===== WINDOWS PROCESS CHECK =====


ProcessId       : 20992
ParentProcessId : 6736
Name            : powershell.exe
CommandLine     : powershell -NoProfile -Command "
                  Get-CimInstance Win32_Process |
                  Where-Object {
                      $_.CommandLine -like \"*40C04A*\" -or
                      $_.CommandLine -like \"*PALMSIM_P00_BASELINE*\"
                  } |
                  Select-Object ProcessId, ParentProcessId, Name, CommandLine |
                  Format-List
                  "





===== CHECKPOINT CHECK =====

40C04A_PALMSIM_P00_BASELINE_CHECKPOINT_SIAK_AOI01_V1.csv
Belum ada.

40C04A_PALMSIM_P00_BASELINE_RESULTS_SIAK_AOI01_V3.csv
Belum ada.


In [2]:
import sys
import json
import time
import warnings
from pathlib import Path

import pandas as pd

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

warnings.filterwarnings("ignore", category=FutureWarning)

# ===== LOAD PALMSIM =====
reg = json.loads(
    (WORKDIR / "40C01B_PALMSIM_SOURCE_REGISTRATION_SIAK_AOI01_V1.json")
    .read_text(encoding="utf-8")
)

working_root = Path(reg["working_root"])
palmsim_master = list(working_root.rglob("palmsim-master"))[0]

for p in [palmsim_master / "PalmSim", palmsim_master]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from palmsim import PalmField

# ===== LOAD THE SAME SMOKE-TEST ASU =====
asu = pd.read_csv(
    WORKDIR / "40C02C_PALMSIM_SIAK_SMOKE_SELECTED_ASU_SIAK_AOI01_V1.csv"
).iloc[0]

weather = pd.read_csv(
    WORKDIR / "40C02C_PALMSIM_SIAK_SMOKE_WEATHER_SIAK_AOI01_V1.csv"
)

weather["DATE"] = pd.to_datetime(weather["DATE"])
weather = weather.set_index("DATE")

print("UNIT_ID :", asu["UNIT_ID"])
print("Days    :", int(asu["DURATION_DAYS"]))
print("\nStarting 1 PALMSIM run...")

# ===== TIMER =====
t0 = time.perf_counter()

p = PalmField(
    year_of_planting=int(asu["YEAR_OF_PLANTING_PROXY"])
)

p.latitude = float(asu["ASU_LATITUDE"])
p.weather.rainfall_series = weather["RAIN_MM_DAY"]
p.weather.radiation_series = weather["RAD_MJ_M2_DAY"]

p.planting_density = 143
p.soil.soil_depth = 1.0

df = p.run(
    duration=int(asu["DURATION_DAYS"])
)

elapsed = time.perf_counter() - t0

print("\n===== RESULT =====")
print("PALMSIM finished ✅")
print("Rows        :", len(df))
print("Columns     :", len(df.columns))
print("Run time    :", round(elapsed, 2), "seconds")
print("Run time min:", round(elapsed / 60, 2), "minutes")

UNIT_ID : SIAK_AOI01_G500_1633,144
Days    : 3650

Starting 1 PALMSIM run...

===== RESULT =====
PALMSIM finished ✅
Rows        : 365
Columns     : 181
Run time    : 7.78 seconds
Run time min: 0.13 minutes


In [3]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40C04A_DIRECT_JUPYTER_P00_BASELINE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40C04A DIRECT-JUPYTER — PALMSIM P00 BASELINE
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD
PALMSIM import: PASS

Input PALMSIM ASUs : 654
Climate-compatible: 654
Excluded          : 0
Climate pairs cached: 12
Recovered checkpoint: 0
Remaining to run    : 654

Starting baseline loop...

Progress: 5/654 (0.8%) | last=16.1s | avg20=15.2s/run | ETA≈164.8 min | fail=0 | elapsed=1.3 min
Progress: 10/654 (1.5%) | last=15.2s | avg20=16.6s/run | ETA≈178.4 min | fail=0 | elapsed=2.8 min
Progress: 15/654 (2.3%) | last=18.2s | avg20=18.0s/run | ETA≈191.4 min | fail=0 | elapsed=4.5 min
Progress: 20/654 (3.1%) | last=6.9s | avg20=16.8s/run | ETA≈177.8 min | fail=0 | elapsed=5.6 min
Progress: 25/654 (3.8%) | last=16.6s | avg20=16.8s/run | ETA≈176.4 min | fail=0 | elapsed=6.9 min
Progress: 30/654 (4.6%) | last=16.8s | avg20=16.8s/run | ETA≈174.9 min | fail=0 | elapsed=8.4 min
Progress: 35/654 (5.4%) | last=15.6s | avg2

In [1]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40C04B_40C05_REPRESENTATIVE_SENSITIVITY_AND_QC_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40C04B + 40C05 — REPRESENTATIVE SENSITIVITY + QC
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD
PALMSIM import: PASS

Frozen baseline:
  ASUs            : 654
  Climate pairs   : 12
  Condition field : CONDITION_SCORE

Representative sensitivity sample:
  Selected ASUs          : 48
  Climate pairs covered  : 12
  Min sample / pair      : 4
  Max sample / pair      : 4
  Features used          : AGE_PROXY_YR, TRAILING_1Y_MEAN_T_HA_YR, CONDITION_SCORE
  Sensitivity scenarios  : 4
  Planned model runs     : 192
  Climate pair cache     : 12
  Recovered runs         : 0
  Remaining runs         : 192

Starting representative sensitivity loop...

Progress: 5/192 (2.6%) | last=5.0s | avg20=6.0s/run | ETA≈18.7 min | fail=0 | elapsed=0.5 min
Progress: 10/192 (5.2%) | last=2.3s | avg20=5.1s/run | ETA≈15.6 min | fail=0 | elapsed=0.9 min
Progress: 15/192 (7.8%) | last=3.1s | avg20=4.4s/run | ETA≈12.9 min | fail=0 

In [2]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D01_APSIM_OILPALM_LOCAL_PREFLIGHT_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40D01 — APSIM OIL PALM LOCAL PREFLIGHT
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

Candidate roots:
 - C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

===== DETECTION =====
Classic executables : 0

NextGen executables : 0

OilPalm model files : 0

OilPalm simulations : 0

40D01 FINISHED
Status: APSIM_OILPALM_NOT_FOUND_LOCALLY
Next  : Install or obtain APSIM Classic 7.10 with its OilPalm model, then rerun this same preflight.
Audit : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D01_APSIM_OILPALM_LOCAL_PREFLIGHT_SIAK_AOI01_V1.json


In [3]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D01B_APSIM_NEXTGEN_OILPALM_LOCAL_DETECTION_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40D01B — APSIM NEXT GENERATION + OILPALM DETECTION
Python : D:\New folder\envs\copernicus_env\python.exe
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

Candidate APSIM roots:
 - C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0

===== APSIM DETECTION =====
NextGen installed : True
Models.exe        : C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\bin\Models.exe
ApsimNG.exe       : C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\bin\ApsimNG.exe
Assemblies        : 2

Version probe:
  returncode: 0
  stdout    : APSIM 2026.8.8099.0

===== OILPALM DETECTION =====
Name hits    : 1
   C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\Examples\OilPalm.apsimx
Content hits : 5
   C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\ApsimNG\Resources\CommonReportVariables\CommonReportingVariables.json
   C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\bin\Models.xml
   C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\E

In [4]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D02A_APSIM_OILPALM_REFERENCE_AUDIT_SMOKE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40D02A — APSIM NEXT GEN OILPALM REFERENCE SMOKE
Python      : D:\New folder\envs\copernicus_env\python.exe
Models.exe  : C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\bin\Models.exe
OilPalm ref : C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\Examples\OilPalm.apsimx
Workdir     : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

===== CLI AUDIT =====
APSIM version:
APSIM 2026.8.8099.0

Enabled simulations:
Simulation

Referenced filenames:
C:/Users/user/AppData/Local/Programs/APSIM2026.8.8099.0/Examples/WeatherFiles/PG_Popondetta.met

APSIMX file version:
195

===== STRUCTURE AUDIT =====
Total model nodes : 80
Simulation nodes  : 2
   {'path': '/Simulations', 'name': 'Simulations', 'enabled': True}
   {'path': '/Simulations/Simulation', 'name': 'Simulation', 'enabled': True}
Clock nodes       : 1
   {'path': '/Simulations/Simulation/Clock', 'name': 'Clock', 'start': '1996-03-01T00:00:00', 'end': '2010-09-29T00:00:00'}
Weather nodes     : 1
   {'path

In [5]:
# ============================================================
# OUTPUT 4 / 40D02B
# APSIM OILPALM — PARAMETER & OUTPUT CONTRACT AUDIT
# SIAK_AOI01
# ============================================================

from pathlib import Path
from datetime import datetime, timedelta
import json
import re

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

APSIM_ROOT = Path(
    r"C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0"
)

OILPALM_FILE = APSIM_ROOT / "Examples" / "OilPalm.apsimx"

OUT_JSON = WORKDIR / (
    "40D02B_APSIM_OILPALM_PARAMETER_OUTPUT_CONTRACT_SIAK_AOI01_V1.json"
)

print("====================================================")
print("40D02B — APSIM OILPALM PARAMETER / OUTPUT CONTRACT")
print("====================================================")
print("OilPalm file:", OILPALM_FILE)

if not OILPALM_FILE.exists():
    raise FileNotFoundError(OILPALM_FILE)

# ------------------------------------------------------------
# READ APSIMX
# ------------------------------------------------------------

with open(OILPALM_FILE, "r", encoding="utf-8-sig") as f:
    apsim = json.load(f)


def walk(node, path=""):
    """Recursively walk APSIM model tree."""
    if isinstance(node, dict):
        name = str(node.get("Name", ""))
        new_path = f"{path}/{name}" if name else path

        yield new_path, node

        for child in node.get("Children", []) or []:
            yield from walk(child, new_path)

    elif isinstance(node, list):
        for item in node:
            yield from walk(item, path)


nodes = list(walk(apsim))


def first_node(type_contains=None, name=None):
    for path, node in nodes:

        t = str(node.get("$type", ""))

        if type_contains and type_contains.lower() not in t.lower():
            continue

        if name and str(node.get("Name", "")).lower() != name.lower():
            continue

        return path, node

    return None, None


# ------------------------------------------------------------
# 1. CORE MODEL NODES
# ------------------------------------------------------------

clock_path, clock = first_node("Models.Clock")
weather_path, weather = first_node("Models.Climate.Weather")
palm_path, palm = first_node("Models.PMF.OilPalm.OilPalm")
manager_path, palm_manager = first_node(
    "Models.Manager",
    "Palm Management"
)
physical_path, physical = first_node("Models.Soils.Physical")
crop_path, palm_soil = first_node(
    "Models.Soils.SoilCrop",
    "OilPalmSoil"
)

annual_path, annual_report = first_node(
    "Models.Report",
    "AnnualOutput"
)

monthly_path, monthly_report = first_node(
    "Models.Report",
    "MonthlyOutput"
)

print("\n===== CORE CONTRACT =====")

print("Clock         :", clock_path)
print("Weather       :", weather_path)
print("OilPalm model :", palm_path)
print("Palm manager  :", manager_path)
print("Soil Physical :", physical_path)
print("OilPalmSoil   :", crop_path)
print("Annual report :", annual_path)
print("Monthly report:", monthly_path)

# ------------------------------------------------------------
# 2. PALM MANAGEMENT PARAMETERS
# ------------------------------------------------------------

manager_params = {}

if palm_manager:
    for p in palm_manager.get("Parameters", []) or []:
        manager_params[p.get("Key")] = p.get("Value")

print("\n===== PALM MANAGEMENT PARAMETERS =====")

for k, v in manager_params.items():
    print(f"{k:15s}: {v}")

expected_manager = {
    "sowing_date",
    "population",
    "cultivar",
    "sowing_depth",
}

missing_manager = sorted(
    expected_manager - set(manager_params.keys())
)

# ------------------------------------------------------------
# 3. DIRECT OILPALM MODEL SETTINGS
# ------------------------------------------------------------

direct_palm_settings = {}

if palm:
    for key in [
        "LAI",
        "UnderstoryCoverMax",
        "UnderstoryLegumeFraction",
        "MaximumRootDepth",
    ]:
        if key in palm:
            direct_palm_settings[key] = palm[key]

print("\n===== DIRECT OILPALM SETTINGS =====")

for k, v in direct_palm_settings.items():
    print(f"{k:28s}: {v}")

# ------------------------------------------------------------
# 4. SOIL CONTRACT
# ------------------------------------------------------------

soil_summary = {}

if physical:

    fields = [
        "Thickness",
        "BD",
        "AirDry",
        "LL15",
        "DUL",
        "SAT",
        "KS",
    ]

    for field in fields:
        value = physical.get(field)

        if isinstance(value, list):
            soil_summary[field] = {
                "layers": len(value),
                "first": value[0] if value else None,
                "last": value[-1] if value else None,
            }


crop_soil_summary = {}

if palm_soil:

    for field in ["LL", "KL", "XF"]:

        value = palm_soil.get(field)

        if isinstance(value, list):
            crop_soil_summary[field] = {
                "layers": len(value),
                "first": value[0] if value else None,
                "last": value[-1] if value else None,
            }

print("\n===== SOIL CONTRACT =====")

for k, v in soil_summary.items():
    print("Physical", k, ":", v)

for k, v in crop_soil_summary.items():
    print("OilPalmSoil", k, ":", v)

# ------------------------------------------------------------
# 5. WEATHER CONTRACT
# ------------------------------------------------------------

weather_filename = weather.get("FileName") if weather else None
weather_path_local = None

if weather_filename:

    resolved = weather_filename.replace(
        "%root%",
        str(APSIM_ROOT)
    )

    resolved = resolved.replace("/", "\\")

    weather_path_local = Path(resolved)

print("\n===== WEATHER CONTRACT =====")

print("Weather reference:", weather_filename)
print("Resolved path    :", weather_path_local)
print(
    "Exists           :",
    weather_path_local.exists()
    if weather_path_local else False
)

weather_columns = []
weather_start = None
weather_end = None
weather_rows = 0

if weather_path_local and weather_path_local.exists():

    lines = weather_path_local.read_text(
        encoding="utf-8",
        errors="ignore"
    ).splitlines()

    header_index = None

    for i, line in enumerate(lines):

        cols = re.split(r"\s+", line.strip().lower())

        if (
            "year" in cols
            and "day" in cols
            and "rain" in cols
            and "radn" in cols
        ):
            weather_columns = cols
            header_index = i
            break

    dates = []

    if header_index is not None:

        year_i = weather_columns.index("year")
        day_i = weather_columns.index("day")

        for line in lines[header_index + 1:]:

            parts = re.split(r"\s+", line.strip())

            if len(parts) < len(weather_columns):
                continue

            try:
                year = int(float(parts[year_i]))
                day = int(float(parts[day_i]))

                dt = (
                    datetime(year, 1, 1)
                    + timedelta(days=day - 1)
                )

                dates.append(dt)
                weather_rows += 1

            except Exception:
                continue

        if dates:
            weather_start = min(dates).date().isoformat()
            weather_end = max(dates).date().isoformat()

print("Columns           :", weather_columns)
print("Rows              :", weather_rows)
print("Start             :", weather_start)
print("End               :", weather_end)

required_weather = {
    "year",
    "day",
    "radn",
    "maxt",
    "mint",
    "rain",
}

missing_weather = sorted(
    required_weather - set(weather_columns)
)

# ------------------------------------------------------------
# 6. OUTPUT CONTRACT
# ------------------------------------------------------------

annual_vars = (
    annual_report.get("VariableNames", [])
    if annual_report else []
)

monthly_vars = (
    monthly_report.get("VariableNames", [])
    if monthly_report else []
)

yield_vars = [
    x for x in annual_vars
    if any(
        key in x.lower()
        for key in [
            "yield",
            "ffb",
            "bunch",
            "harvest",
        ]
    )
]

print("\n===== ANNUAL YIELD OUTPUT CANDIDATES =====")

for x in yield_vars:
    print(" ", x)

annual_yield_found = any(
    "annualyield" in x.lower()
    for x in annual_vars
)

# ------------------------------------------------------------
# 7. OTHER MANAGEMENT NODES
# ------------------------------------------------------------

managers = []

for path, node in nodes:

    if "Models.Manager" not in str(node.get("$type", "")):
        continue

    params = {
        p.get("Key"): p.get("Value")
        for p in node.get("Parameters", []) or []
    }

    managers.append({
        "path": path,
        "name": node.get("Name"),
        "parameters": params,
    })

print("\n===== MANAGER NODES =====")

for m in managers:
    print(
        f"{m['name']} | "
        f"{len(m['parameters'])} exposed parameter(s)"
    )

# ------------------------------------------------------------
# 8. FINAL CONTRACT GATE
# ------------------------------------------------------------

checks = {
    "oilpalm_model_found": palm is not None,
    "clock_found": clock is not None,
    "weather_found": weather is not None,
    "weather_file_exists":
        weather_path_local.exists()
        if weather_path_local else False,
    "weather_required_columns_complete":
        len(missing_weather) == 0,
    "palm_management_found":
        palm_manager is not None,
    "palm_management_required_params_complete":
        len(missing_manager) == 0,
    "soil_physical_found":
        physical is not None,
    "oilpalm_soil_crop_found":
        palm_soil is not None,
    "annual_output_found":
        annual_report is not None,
    "annual_yield_output_found":
        annual_yield_found,
}

status = (
    "PASS_CONTRACT_READY"
    if all(checks.values())
    else "REVIEW"
)

report = {
    "gate": "40D02B_APSIM_OILPALM_PARAMETER_OUTPUT_CONTRACT",
    "status": status,

    "clock": {
        "Start": clock.get("Start") if clock else None,
        "End": clock.get("End") if clock else None,
    },

    "weather": {
        "reference": weather_filename,
        "resolved_path":
            str(weather_path_local)
            if weather_path_local else None,
        "columns": weather_columns,
        "rows": weather_rows,
        "start": weather_start,
        "end": weather_end,
        "missing_required_columns": missing_weather,
    },

    "palm_management_parameters":
        manager_params,

    "missing_palm_management_parameters":
        missing_manager,

    "direct_oilpalm_settings":
        direct_palm_settings,

    "soil_physical":
        soil_summary,

    "oilpalm_soil_crop":
        crop_soil_summary,

    "annual_output_variables":
        annual_vars,

    "monthly_output_variables":
        monthly_vars,

    "yield_output_candidates":
        yield_vars,

    "manager_nodes":
        managers,

    "checks":
        checks,

    "contract_classification": {

        "DIRECT_SCENARIO_INPUTS": [
            "Weather.FileName",
            "Clock.Start",
            "Clock.End",
            "Palm Management.sowing_date",
            "Palm Management.population",
            "Palm Management.cultivar",
            "Palm Management.sowing_depth",
        ],

        "CONDITIONAL_INPUTS": [
            "Soil.Physical profile",
            "OilPalmSoil LL/KL/XF",
            "MaximumRootDepth",
            "fertiliser management",
            "irrigation management",
        ],

        "PRIMARY_BENCHMARK_OUTPUT": (
            "[Calculations].Script.AnnualYield"
        ),

        "INTERPRETATION": (
            "APSIM OilPalm is used as a representative "
            "process/scenario benchmark, not as locally "
            "validated actual FFB yield."
        ),
    },
}

OUT_JSON.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n====================================================")
print("40D02B FINISHED")
print("====================================================")

for k, v in checks.items():
    print(f"{k:45s}: {v}")

print()
print("Missing manager parameters:", missing_manager)
print("Missing weather columns    :", missing_weather)

print()
print("Status :", status)
print("Audit  :", OUT_JSON)

if status == "PASS_CONTRACT_READY":
    print(
        "Next   : 40D03 Siak representative "
        "APSIM scenario design + weather adapter."
    )
else:
    print(
        "Next   : Review failed contract items "
        "before Siak adaptation."
    )

print("====================================================")

40D02B — APSIM OILPALM PARAMETER / OUTPUT CONTRACT
OilPalm file: C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\Examples\OilPalm.apsimx

===== CORE CONTRACT =====
Clock         : /Simulations/Simulation/Clock
Weather       : /Simulations/Simulation/Weather
OilPalm model : /Simulations/Simulation/Block/OilPalm
Palm manager  : /Simulations/Simulation/Block/Palm Management
Soil Physical : /Simulations/Simulation/Block/Soil/Physical
OilPalmSoil   : /Simulations/Simulation/Block/Soil/Physical/OilPalmSoil
Annual report : /Simulations/Simulation/Block/AnnualOutput
Monthly report: /Simulations/Simulation/Block/MonthlyOutput

===== PALM MANAGEMENT PARAMETERS =====
sowing_date    : 03/01/1996 00:00:00
population     : 135
cultivar       : Dami
sowing_depth   : 200

===== DIRECT OILPALM SETTINGS =====
LAI                         : 0.0
UnderstoryCoverMax          : 0.4
UnderstoryLegumeFraction    : 1.0
MaximumRootDepth            : 5000.0

===== SOIL CONTRACT =====
Physical Thickness : {'

In [6]:
# ============================================================
# OUTPUT 4 / 40D03A
# SIAK CLIMATE SOURCE SCHEMA AUDIT FOR APSIM ADAPTER
# ============================================================

from pathlib import Path
import pandas as pd

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

print("====================================================")
print("40D03A — SIAK CLIMATE SOURCE SCHEMA AUDIT")
print("====================================================")
print("Workdir:", WORKDIR)

# Cari CSV yang kemungkinan berasal dari 40B climate preparation.
keywords = [
    "CHIRPS",
    "ERA5",
    "ERA5L",
    "CLIMATE",
    "WEATHER",
]

candidates = []

for f in WORKDIR.glob("*.csv"):
    name = f.name.upper()

    if any(k in name for k in keywords):
        candidates.append(f)

candidates = sorted(candidates)

print("\nCandidate climate CSV files:", len(candidates))

for i, f in enumerate(candidates, 1):

    print("\n----------------------------------------------------")
    print(f"[{i}] {f.name}")

    try:
        df = pd.read_csv(f, nrows=5)

        print("Columns:")
        print(list(df.columns))

        print("\nFirst rows:")
        print(df.head(3).to_string(index=False))

    except Exception as e:
        print("READ ERROR:", repr(e))

print("\n====================================================")
print("40D03A FINISHED")
print("====================================================")
print(
    "Next: identify exact CHIRPS rainfall + ERA5-Land "
    "radiation/Tmax/Tmin files for APSIM .met generation."
)
print("====================================================")

40D03A — SIAK CLIMATE SOURCE SCHEMA AUDIT
Workdir: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

Candidate climate CSV files: 17

----------------------------------------------------
[1] 40B04A_ASU_CLIMATE_CELL_MAPPING_SIAK_AOI01_V1.csv
Columns:
['UNIT_ID', 'GRID_INDEX', 'GRID_SIZE_M', 'CELL_AREA_HA', 'ASU_AREA_HA', 'PLANTATION_FRACTION', 'AGE_DATA_VALID', 'AGE_P10', 'AGE_P50', 'AGE_P90', 'AGE_RANGE_P90_P10', 'AGE_VALID_FRACTION', 'CONDITION_DATA_VALID', 'COND_P10', 'COND_P50', 'COND_P90', 'CONDITION_RANGE_P90_P10', 'CONDITION_VALID_FRACTION', 'REFERENCE_YEAR', 'ASU_STAGE', 'ASU_AREA_HA_EXACT', 'PLANTATION_FRACTION_EXACT', 'CENTROID_LON_EXACT', 'CENTROID_LAT_EXACT', 'ASU_TYPE_FINAL', 'ASU_VERSION_FINAL', 'GEOMETRY_STATUS', 'SAND_PCT_0_100_MEAN', 'SAND_PCT_0_100_STD', 'SAND_PCT_0_100_COUNT', 'SILT_PCT_0_100_MEAN', 'SILT_PCT_0_100_STD', 'SILT_PCT_0_100_COUNT', 'CLAY_PCT_0_100_MEAN', 'CLAY_PCT_0_100_STD', 'CLAY_PCT_0_100_COUNT', 'TEXTURE_SUM_PCT', 'SOIL_TEXTURE_DATA_

In [7]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D03B_APSIM_SIAK_WEATHER_ADAPTER_1ASU_SMOKE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40D03B — APSIM SIAK WEATHER ADAPTER + 1-ASU SMOKE
Target ASU : SIAK_AOI01_G500_1633,144
Workdir    : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD

===== TARGET CONTRACT =====
ASU                 : SIAK_AOI01_G500_1633,144
Age proxy (P50)     : 10 yr
Reference year      : 2025
Planting year proxy : 2015
CHIRPS cell         : CHIRPS_R986_C5636
ERA5-Land cell      : ERA5L_+00.7_+101.8
Centroid lat/lon    : 0.6539915932240334 101.84554408110968
Simulation          : 2015-01-01 to 2025-12-31

===== WEATHER ADAPTER =====
Rows     : 9131
Start    : 2001-01-01
End      : 2025-12-31
TAV      : 26.0266 °C
AMP      : 1.2407 °C
Rain mean: 7.1598 mm/day
Rad mean : 16.2405 MJ/m²/day
MET file : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D03B_APSIM_SIAK_1ASU_SMOKE_SIAK_AOI01_V1\SIAK_AOI01_APSIM_WEATHER_2001_2025_V1.met

===== APSIM ADAPTATION =====
Clock.Start : 2015-01-01T00:00:00
Clock.End   : 2025-12-31T00:00:00
Weather     : C:/Users/user/Documents/K

In [8]:
# ============================================================
# 40D03B-QC1
# DIAGNOSE APSIM OILPALM ANNUALYIELD = NULL
# ============================================================

from pathlib import Path
import pandas as pd
import json

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

APSIM_ROOT = Path(
    r"C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0"
)

OFFICIAL_APSIMX = APSIM_ROOT / "Examples" / "OilPalm.apsimx"

OFFICIAL_CSV = (
    WORKDIR
    / "40D02A_APSIM_OILPALM_REFERENCE_SMOKE_SIAK_AOI01_V1"
    / "RUN1"
    / "OilPalm_REFERENCE_8099.AnnualOutput.csv"
)

SIAK_CSV = (
    WORKDIR
    / "40D03B_APSIM_SIAK_1ASU_SMOKE_SIAK_AOI01_V1"
    / "RUN1"
    / "OilPalm_SIAK_1ASU_SMOKE_V1.AnnualOutput.csv"
)

print("====================================================")
print("40D03B-QC1 — ANNUAL YIELD DIAGNOSTIC")
print("====================================================")

# ------------------------------------------------------------
# 1. PRINT RELEVANT OUTPUTS
# ------------------------------------------------------------

candidate_terms = [
    "Year",
    "Age",
    "Yield",
    "Bunch",
    "Rain",
    "LAI",
]

for label, path in [
    ("OFFICIAL REFERENCE", OFFICIAL_CSV),
    ("SIAK", SIAK_CSV),
]:
    print("\n====================================================")
    print(label)
    print("====================================================")

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    df = pd.read_csv(path)

    cols = [
        c for c in df.columns
        if any(term.lower() in c.lower() for term in candidate_terms)
    ]

    print("Rows:", len(df))
    print("\nRelevant columns:")
    for c in cols:
        print(
            f"{c:45s} "
            f"non-null={df[c].notna().sum():3d}"
        )

    print("\nRelevant annual table:")
    print(df[cols].to_string(index=False))


# ------------------------------------------------------------
# 2. INSPECT 'CALCULATIONS' MANAGER CODE
# ------------------------------------------------------------

with open(OFFICIAL_APSIMX, "r", encoding="utf-8-sig") as f:
    apsim = json.load(f)


def walk(node):
    if isinstance(node, dict):
        yield node
        for child in node.get("Children", []) or []:
            yield from walk(child)
    elif isinstance(node, list):
        for child in node:
            yield from walk(child)


calc = None

for node in walk(apsim):
    if (
        node.get("$type") == "Models.Manager, Models"
        and node.get("Name") == "Calculations"
    ):
        calc = node
        break

print("\n====================================================")
print("CALCULATIONS MANAGER — RELEVANT CODE")
print("====================================================")

if calc is None:
    print("Calculations manager NOT FOUND.")
else:
    code = calc.get("Code")

    if not code:
        code_array = calc.get("CodeArray", [])
        code = "\n".join(code_array)

    lines = code.splitlines()

    keywords = [
        "AnnualYield",
        "HarvestFFB",
        "AnnualBunch",
        "EndOfYear",
        "StartOfYear",
        "DoManagement",
        "Clock",
    ]

    for i, line in enumerate(lines, 1):
        if any(k.lower() in line.lower() for k in keywords):
            lo = max(0, i - 3)
            hi = min(len(lines), i + 3)

            print(f"\n--- around line {i} ---")

            for j in range(lo, hi):
                print(f"{j+1:4d}: {lines[j]}")

print("\n====================================================")
print("40D03B-QC1 FINISHED")
print("====================================================")

40D03B-QC1 — ANNUAL YIELD DIAGNOSTIC

OFFICIAL REFERENCE


ParserError: Error tokenizing data. C error: Expected 43 fields in line 3, saw 65


In [9]:
# ============================================================
# 40D03B-QC1A
# INSPECT RAW APSIM CSV STRUCTURE
# ============================================================

from pathlib import Path
import csv

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

files = {
    "OFFICIAL": (
        WORKDIR
        / "40D02A_APSIM_OILPALM_REFERENCE_SMOKE_SIAK_AOI01_V1"
        / "RUN1"
        / "OilPalm_REFERENCE_8099.AnnualOutput.csv"
    ),
    "SIAK": (
        WORKDIR
        / "40D03B_APSIM_SIAK_1ASU_SMOKE_SIAK_AOI01_V1"
        / "RUN1"
        / "OilPalm_SIAK_1ASU_SMOKE_V1.AnnualOutput.csv"
    ),
}

print("====================================================")
print("40D03B-QC1A — RAW CSV STRUCTURE")
print("====================================================")

for label, path in files.items():

    print(f"\n===== {label} =====")
    print("File:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        reader = csv.reader(f)

        for i, row in enumerate(reader, start=1):

            print(f"\nLINE {i} | FIELD COUNT = {len(row)}")
            print(row)

            if i >= 6:
                break

print("\n====================================================")
print("40D03B-QC1A FINISHED")
print("====================================================")

40D03B-QC1A — RAW CSV STRUCTURE

===== OFFICIAL =====
File: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D02A_APSIM_OILPALM_REFERENCE_SMOKE_SIAK_AOI01_V1\RUN1\OilPalm_REFERENCE_8099.AnnualOutput.csv
Exists: True

LINE 1 | FIELD COUNT = 36
['SimulationName', 'SimulationID', 'AnnualAverageNO3', 'AnnualBunchNPP', 'AnnualDenitrification', 'AnnualET', 'AnnualFertiliserNAdded', 'AnnualFrondNPP', 'AnnualIrrigation', 'AnnualLitterMass', 'AnnualLitterN', 'AnnualN2O', 'AnnualNLeaching', 'AnnualPalmNUptake', 'AnnualPalmVegetativeNPP', 'AnnualPotentialEvaporation', 'AnnualRain', 'AnnualRootNPP', 'AnnualStemNPP', 'AnnualTotalNInput', 'AnnualTotalVegetativeNPP', 'AnnualUnderstoryNFixation', 'AnnualUnderstoryNPP', 'AnnualUnderstoryNUptake', 'Calculations.Script.AnnualBunches', 'Calculations.Script.AnnualBunchSize', 'Calculations.Script.AnnualFronds', 'Calculations.Script.AnnualNRemoved', 'Calculations.Script.AnnualYield', 'CheckpointID', 'CheckpointName', 'Clock.Today', 'Clock

In [10]:
# ============================================================
# 40D03B-QC1B
# APSIM SQLITE DATABASE OUTPUT DIAGNOSTIC
# ============================================================

from pathlib import Path
import sqlite3
import pandas as pd

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

DBS = {
    "OFFICIAL": (
        WORKDIR
        / "40D02A_APSIM_OILPALM_REFERENCE_SMOKE_SIAK_AOI01_V1"
        / "RUN1"
        / "OilPalm_REFERENCE_8099.db"
    ),
    "SIAK": (
        WORKDIR
        / "40D03B_APSIM_SIAK_1ASU_SMOKE_SIAK_AOI01_V1"
        / "RUN1"
        / "OilPalm_SIAK_1ASU_SMOKE_V1.db"
    ),
}

print("====================================================")
print("40D03B-QC1B — APSIM SQLITE OUTPUT DIAGNOSTIC")
print("====================================================")

for label, db_path in DBS.items():

    print(f"\n{'='*60}")
    print(label)
    print("="*60)
    print("DB:", db_path)
    print("Exists:", db_path.exists())

    if not db_path.exists():
        continue

    con = sqlite3.connect(db_path)

    # --------------------------------------------------------
    # List tables
    # --------------------------------------------------------

    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        ORDER BY name
        """,
        con
    )["name"].tolist()

    print("\nTables:")
    print(tables)

    annual_tables = [
        t for t in tables
        if "annualoutput" in t.lower()
    ]

    if not annual_tables:
        print("AnnualOutput table NOT FOUND.")
        con.close()
        continue

    table = annual_tables[0]

    # --------------------------------------------------------
    # Read actual SQLite schema
    # --------------------------------------------------------

    schema = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        con
    )

    cols = schema["name"].tolist()

    relevant = [
        c for c in cols
        if any(
            key in c.lower()
            for key in [
                "yield",
                "bunch",
                "rain",
                "age",
                "lai",
                "today",
                "year",
            ]
        )
    ]

    print("\nAnnualOutput table:", table)

    print("\nRelevant columns:")
    for c in relevant:
        dtype = schema.loc[
            schema["name"] == c, "type"
        ].iloc[0]

        print(f"  {c} | SQLite type = {dtype}")

    # --------------------------------------------------------
    # Read real values directly from SQLite
    # --------------------------------------------------------

    quoted = ", ".join(
        '"' + c.replace('"', '""') + '"'
        for c in relevant
    )

    df = pd.read_sql_query(
        f'SELECT {quoted} FROM "{table}"',
        con
    )

    print("\nAnnual data:")
    print(df.to_string(index=False))

    # --------------------------------------------------------
    # AnnualYield QC
    # --------------------------------------------------------

    yield_cols = [
        c for c in df.columns
        if "annualyield" in c.lower()
    ]

    print("\nAnnualYield QC:")

    if not yield_cols:
        print("  AnnualYield column NOT FOUND")

    else:
        ycol = yield_cols[0]

        y = pd.to_numeric(
            df[ycol],
            errors="coerce"
        )

        print("  Column   :", ycol)
        print("  Rows     :", len(y))
        print("  Non-null :", y.notna().sum())
        print("  Null     :", y.isna().sum())

        if y.notna().any():
            print("  Min      :", y.min())
            print("  Median   :", y.median())
            print("  Max      :", y.max())
            print("  Last     :", y.dropna().iloc[-1])

    con.close()

print("\n====================================================")
print("40D03B-QC1B FINISHED")
print("====================================================")

40D03B-QC1B — APSIM SQLITE OUTPUT DIAGNOSTIC

OFFICIAL
DB: C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D02A_APSIM_OILPALM_REFERENCE_SMOKE_SIAK_AOI01_V1\RUN1\OilPalm_REFERENCE_8099.db
Exists: True

Tables:
['AnnualOutput', 'MonthlyOutput', '_Checkpoints', '_InitialConditions', '_Messages', '_Simulations', '_Units']

AnnualOutput table: AnnualOutput

Relevant columns:
  Clock.Today | SQLite type = date
  Clock.Today.Year | SQLite type = INTEGER
  OilPalm.Age | SQLite type = REAL
  OilPalm.LAI | SQLite type = REAL
  AnnualRain | SQLite type = REAL
  Calculations.Script.AnnualBunches | SQLite type = REAL
  Calculations.Script.AnnualYield | SQLite type = REAL
  Calculations.Script.AnnualBunchSize | SQLite type = REAL
  AnnualBunchNPP | SQLite type = REAL
  AnnualAverageNO3 | SQLite type = REAL

Annual data:
        Clock.Today  Clock.Today.Year  OilPalm.Age  OilPalm.LAI  AnnualRain  Calculations.Script.AnnualBunches  Calculations.Script.AnnualYield  Calculations.Scr

In [11]:
# ============================================================
# 40D03B-QC2
# VERIFY APSIM ANNUALYIELD UNIT FROM SQLITE
# ============================================================

from pathlib import Path
import sqlite3
import pandas as pd

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

DB = (
    WORKDIR
    / "40D03B_APSIM_SIAK_1ASU_SMOKE_SIAK_AOI01_V1"
    / "RUN1"
    / "OilPalm_SIAK_1ASU_SMOKE_V1.db"
)

print("====================================================")
print("40D03B-QC2 — APSIM ANNUALYIELD UNIT AUDIT")
print("====================================================")

con = sqlite3.connect(DB)

units = pd.read_sql_query(
    'SELECT * FROM "_Units"',
    con
)

con.close()

print("Columns:", list(units.columns))

hits = units[
    units.astype(str)
         .apply(
             lambda row:
             row.str.contains(
                 "AnnualYield",
                 case=False,
                 regex=False
             ).any(),
             axis=1
         )
]

print("\nAnnualYield unit record:")
print(hits.to_string(index=False))

print("\n====================================================")
print("40D03B-QC2 FINISHED")
print("====================================================")

40D03B-QC2 — APSIM ANNUALYIELD UNIT AUDIT
Columns: ['TableName', 'ColumnHeading', 'Units']

AnnualYield unit record:
Empty DataFrame
Columns: [TableName, ColumnHeading, Units]
Index: []

40D03B-QC2 FINISHED


In [12]:
# ============================================================
# OUTPUT 4 / 40D03C
# APSIM REPRESENTATIVE BENCHMARK DESIGN
# SIAK_AOI01
# ============================================================

from pathlib import Path
import pandas as pd
import json

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

SAMPLE = WORKDIR / (
    "40C04B_REPRESENTATIVE_SENSITIVITY_SAMPLE_SIAK_AOI01_V1.csv"
)

MAPPING = WORKDIR / (
    "40B04A_ASU_CLIMATE_CELL_MAPPING_SIAK_AOI01_V1.csv"
)

PAIRMAP = WORKDIR / (
    "40B04C_LITE_ASU_CLIMATE_PAIR_MAPPING_SIAK_AOI01_V1.csv"
)

OUT_CSV = WORKDIR / (
    "40D03C_APSIM_REPRESENTATIVE_BENCHMARK_DESIGN_SIAK_AOI01_V1.csv"
)

OUT_JSON = WORKDIR / (
    "40D03C_APSIM_REPRESENTATIVE_BENCHMARK_DESIGN_QC_SIAK_AOI01_V1.json"
)

print("====================================================")
print("40D03C — APSIM REPRESENTATIVE BENCHMARK DESIGN")
print("====================================================")

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

sample = pd.read_csv(SAMPLE)
mapping = pd.read_csv(MAPPING)
pairmap = pd.read_csv(PAIRMAP)

if "UNIT_ID" not in sample.columns:
    raise ValueError("UNIT_ID missing from representative sample.")

if sample["UNIT_ID"].duplicated().any():
    raise ValueError("Duplicate UNIT_ID in representative sample.")

# Keep authoritative ASU attributes from 40B.
cols = [
    "UNIT_ID",
    "AGE_P50",
    "REFERENCE_YEAR",
    "CENTROID_LON_EXACT",
    "CENTROID_LAT_EXACT",
    "CHIRPS_CELL_ID",
    "ERA5L_CELL_ID",
]

base = sample[["UNIT_ID"]].merge(
    mapping[cols],
    on="UNIT_ID",
    how="left",
    validate="one_to_one"
)

base = base.merge(
    pairmap[["UNIT_ID", "CLIMATE_PAIR_ID"]],
    on="UNIT_ID",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# APSIM CONTRACT
# ------------------------------------------------------------

base["AGE_PROXY_YR"] = base["AGE_P50"].round().astype("Int64")

base["PLANTING_YEAR_PROXY"] = (
    base["REFERENCE_YEAR"] - base["AGE_PROXY_YR"]
).astype("Int64")

base["SIM_START"] = (
    base["PLANTING_YEAR_PROXY"].astype(str) + "-01-01"
)

base["SIM_END"] = "2025-12-31"

# Same baseline density as PALMSIM.
base["POPULATION_PALM_HA"] = 143

# APSIM official reference settings retained.
base["CULTIVAR"] = "Dami"
base["SOIL_POLICY"] = "OFFICIAL_APSIM_OILPALM_REFERENCE"
base["MANAGEMENT_POLICY"] = "OFFICIAL_APSIM_REFERENCE"
base["IRRIGATION_POLICY"] = "OFFICIAL_REFERENCE_SETTING"
base["OUTPUT_FIELD"] = "Calculations.Script.AnnualYield"
base["OUTPUT_UNIT"] = "t_FFB_ha_yr"

# ------------------------------------------------------------
# QC
# ------------------------------------------------------------

missing = base[
    [
        "AGE_PROXY_YR",
        "PLANTING_YEAR_PROXY",
        "CHIRPS_CELL_ID",
        "ERA5L_CELL_ID",
        "CLIMATE_PAIR_ID",
    ]
].isna().sum()

pair_counts = (
    base.groupby("CLIMATE_PAIR_ID")
        .size()
        .sort_index()
)

planting_min = int(base["PLANTING_YEAR_PROXY"].min())
planting_max = int(base["PLANTING_YEAR_PROXY"].max())

checks = {
    "representative_asu_count_48": len(base) == 48,
    "unique_asu_48": base["UNIT_ID"].nunique() == 48,
    "climate_pair_count_12": base["CLIMATE_PAIR_ID"].nunique() == 12,
    "four_asu_per_climate_pair": bool((pair_counts == 4).all()),
    "no_missing_contract_fields": int(missing.sum()) == 0,
    "all_simulations_supported_by_2001_weather":
        planting_min >= 2001,
}

status = (
    "PASS_DESIGN_FROZEN"
    if all(checks.values())
    else "REVIEW"
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

base.to_csv(
    OUT_CSV,
    index=False
)

qc = {
    "gate": "40D03C_APSIM_REPRESENTATIVE_BENCHMARK_DESIGN",
    "status": status,
    "representative_asus": int(len(base)),
    "climate_pairs": int(base["CLIMATE_PAIR_ID"].nunique()),
    "pair_counts": pair_counts.to_dict(),
    "planting_year_min": planting_min,
    "planting_year_max": planting_max,
    "population_palm_ha": 143,
    "cultivar": "Dami",
    "soil_policy": "OFFICIAL_APSIM_OILPALM_REFERENCE",
    "output_field": "Calculations.Script.AnnualYield",
    "output_unit": "t_FFB_ha_yr",
    "checks": checks,
}

OUT_JSON.write_text(
    json.dumps(qc, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("Representative ASUs :", len(base))
print("Climate pairs       :", base["CLIMATE_PAIR_ID"].nunique())

print("\nASUs per climate pair:")
print(pair_counts.to_string())

print("\nPlanting-year proxy :", planting_min, "to", planting_max)

print("\nQC:")
for k, v in checks.items():
    print(f"{k:45s}: {v}")

print("\n====================================================")
print("40D03C FINISHED")
print("====================================================")
print("Status :", status)
print("CSV    :", OUT_CSV)
print("QC     :", OUT_JSON)

if status == "PASS_DESIGN_FROZEN":
    print("Next   : 40D04 APSIM representative benchmark execution.")
else:
    print("Next   : Review design/QC before running APSIM.")

print("====================================================")

40D03C — APSIM REPRESENTATIVE BENCHMARK DESIGN
Representative ASUs : 48
Climate pairs       : 12

ASUs per climate pair:
CLIMATE_PAIR_ID
CHIRPS_R984_C5634__ERA5L_+00.8_+101.7    4
CHIRPS_R984_C5635__ERA5L_+00.8_+101.8    4
CHIRPS_R984_C5636__ERA5L_+00.8_+101.8    4
CHIRPS_R984_C5637__ERA5L_+00.8_+101.9    4
CHIRPS_R985_C5634__ERA5L_+00.7_+101.7    4
CHIRPS_R985_C5635__ERA5L_+00.7_+101.8    4
CHIRPS_R985_C5636__ERA5L_+00.7_+101.8    4
CHIRPS_R985_C5637__ERA5L_+00.7_+101.9    4
CHIRPS_R986_C5634__ERA5L_+00.7_+101.7    4
CHIRPS_R986_C5635__ERA5L_+00.7_+101.8    4
CHIRPS_R986_C5636__ERA5L_+00.7_+101.8    4
CHIRPS_R986_C5637__ERA5L_+00.7_+101.9    4

Planting-year proxy : 2001 to 2023

QC:
representative_asu_count_48                  : True
unique_asu_48                                : True
climate_pair_count_12                        : True
four_asu_per_climate_pair                    : True
no_missing_contract_fields                   : True
all_simulations_supported_by_2001_weather    :

In [13]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04A_APSIM_REPRESENTATIVE_BENCHMARK_EXECUTION_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40D04A — APSIM REPRESENTATIVE BENCHMARK EXECUTION
Models.exe : C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\bin\Models.exe
Design     : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D03C_APSIM_REPRESENTATIVE_BENCHMARK_DESIGN_SIAK_AOI01_V1.csv
Run root   : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04A_APSIM_REPRESENTATIVE_RUNS_SIAK_AOI01_V1

Loading climate tables...
Climate pairs cached: 12

[01/48] RUN | SIAK_AOI01_G500_1609,166
  FAIL: ValueError('Age-match difference too large: 18.000000 yr')

[02/48] RUN | SIAK_AOI01_G500_1609,167
  FAIL: ValueError('Age-match difference too large: 18.000000 yr')

[03/48] RUN | SIAK_AOI01_G500_1609,171
  PASS | age-match=9.0082 (diff=0.0082) | yield=33.8505 t FFB/ha/yr | 5.9s

[04/48] RUN | SIAK_AOI01_G500_1611,170
  PASS | age-match=12.0082 (diff=0.0082) | yield=19.5026 t FFB/ha/yr | 6.0s

[05/48] RUN | SIAK_AOI01_G500_1612,172
  PASS | age-match=4.0027 (diff=0.0027) | yield=4.0413 t FF

In [14]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04A_V2_APSIM_REPRESENTATIVE_BENCHMARK_CORRECTED_SIAK_AOI01.py",
    encoding="utf-8"
).read())

40D04A-V2 — APSIM REPRESENTATIVE BENCHMARK
CORRECTED: FIXED 2007 ENDCROP DISABLED
Models.exe : C:\Users\user\AppData\Local\Programs\APSIM2026.8.8099.0\bin\Models.exe
Design     : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D03C_APSIM_REPRESENTATIVE_BENCHMARK_DESIGN_SIAK_AOI01_V1.csv
Run root   : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04A_V2_APSIM_REPRESENTATIVE_RUNS_SIAK_AOI01

Loading climate tables...
Climate pairs cached: 12
Fixed EndCrop code lines disabled: 2
Installed APSIM example remains untouched.

[01/48] RUN | SIAK_AOI01_G500_1609,166
  PASS | plant=2005 | age-match=20.0137 (diff=0.0137) | yield=18.8181 t FFB/ha/yr | 8.4s

[02/48] RUN | SIAK_AOI01_G500_1609,167
  PASS | plant=2006 | age-match=19.0137 (diff=0.0137) | yield=18.0904 t FFB/ha/yr | 9.0s

[03/48] RUN | SIAK_AOI01_G500_1609,171
  PASS | plant=2016 | age-match=9.0082 (diff=0.0082) | yield=33.8505 t FFB/ha/yr | 6.6s

[04/48] RUN | SIAK_AOI01_G500_1611,170
  PASS

In [15]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04B_APSIM_BASELINE_DIAGNOSTICS_POPULATION_SENSITIVITY_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40D04B — APSIM BASELINE DIAGNOSTICS + POP SENSITIVITY
Baseline : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04A_V2_APSIM_REPRESENTATIVE_BENCHMARK_RESULTS_SIAK_AOI01.csv
Run root : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40D04B_APSIM_POPULATION_SENSITIVITY_RUNS_SIAK_AOI01_V1

[001/96] RUN | A130_POP_LOW (130/ha) | SIAK_AOI01_G500_1609,146
  PASS | yield=19.7273 | delta=+0.6424 | 16.7s

[002/96] RUN | A150_POP_HIGH (150/ha) | SIAK_AOI01_G500_1609,146
  PASS | yield=17.9389 | delta=-1.1459 | 21.3s

[003/96] RUN | A130_POP_LOW (130/ha) | SIAK_AOI01_G500_1609,151
  PASS | yield=20.0571 | delta=+1.7594 | 23.5s

[004/96] RUN | A150_POP_HIGH (150/ha) | SIAK_AOI01_G500_1609,151
  PASS | yield=18.5020 | delta=+0.2043 | 21.2s

[005/96] RUN | A130_POP_LOW (130/ha) | SIAK_AOI01_G500_1609,157
  PASS | yield=21.6252 | delta=-0.2906 | 23.6s

[006/96] RUN | A150_POP_HIGH (150/ha) | SIAK_AOI01_G500_1609,157
  PASS | yield=22.2097 | delta=+0.2938 | 24

In [16]:
# ============================================================
# OUTPUT 4 / 40D05
# APSIM OILPALM BENCHMARK — FINAL QC & FREEZE
# SIAK_AOI01
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

BASELINE_CSV = WORKDIR / (
    "40D04A_V2_APSIM_REPRESENTATIVE_BENCHMARK_RESULTS_SIAK_AOI01.csv"
)

SENS_CSV = WORKDIR / (
    "40D04B_APSIM_POPULATION_SENSITIVITY_RESULTS_SIAK_AOI01_V1.csv"
)

OUT_CSV = WORKDIR / (
    "40D05_APSIM_REPRESENTATIVE_BENCHMARK_FROZEN_SIAK_AOI01_V1.csv"
)

OUT_JSON = WORKDIR / (
    "40D05_APSIM_BENCHMARK_FREEZE_QC_SIAK_AOI01_V1.json"
)

TOL = 1e-8

print("====================================================")
print("40D05 — APSIM BENCHMARK FINAL QC / FREEZE")
print("====================================================")

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

base = pd.read_csv(BASELINE_CSV)
sens = pd.read_csv(SENS_CSV)

print("Baseline rows    :", len(base))
print("Sensitivity rows :", len(sens))

# ------------------------------------------------------------
# 2. BASIC CONTRACT QC
# ------------------------------------------------------------

if len(base) != 48:
    raise ValueError(f"Expected 48 baseline ASUs; found {len(base)}")

if base["UNIT_ID"].nunique() != 48:
    raise ValueError("Baseline UNIT_ID is not unique.")

if not (base["STATUS"] == "PASS").all():
    raise ValueError("Baseline contains non-PASS rows.")

if len(sens) != 96:
    raise ValueError(f"Expected 96 sensitivity rows; found {len(sens)}")

if sens[
    ["UNIT_ID", "SCENARIO"]
].drop_duplicates().shape[0] != 96:
    raise ValueError("Sensitivity UNIT_ID/scenario pairs not unique.")

# ------------------------------------------------------------
# 3. CANONICAL BASELINE
# ------------------------------------------------------------

base_keep = base[
    [
        "UNIT_ID",
        "AGE_PROXY_YR",
        "PLANTING_YEAR_PROXY",
        "CLIMATE_PAIR_ID",
        "CHIRPS_CELL_ID",
        "ERA5L_CELL_ID",
        "CENTROID_LAT_EXACT",
        "CENTROID_LON_EXACT",
        "POPULATION_PALM_HA",
        "CULTIVAR",
        "SOIL_POLICY",
        "MANAGEMENT_POLICY",
        "ENDCROP_POLICY",
        "age_matched_model_age_yr",
        "age_match_abs_diff_yr",
        "age_matched_calendar_year",
        "age_matched_annual_yield_t_ha_yr",
        "age_matched_trailing3y_mean_yield_t_ha_yr",
        "age_matched_lai",
        "age_matched_annual_rain_mm",
    ]
].copy()

base_keep = base_keep.rename(
    columns={
        "age_matched_annual_yield_t_ha_yr":
            "APSIM_A143_BASE_T_FFB_HA_YR",

        "age_matched_trailing3y_mean_yield_t_ha_yr":
            "APSIM_A143_TRAILING3Y_T_FFB_HA_YR",
    }
)

# ------------------------------------------------------------
# 4. PIVOT SENSITIVITY
# ------------------------------------------------------------

sens_yield = sens.pivot(
    index="UNIT_ID",
    columns="SCENARIO",
    values="YIELD_T_FFB_HA_YR",
).reset_index()

sens_yield = sens_yield.rename(
    columns={
        "A130_POP_LOW":
            "APSIM_A130_T_FFB_HA_YR",

        "A150_POP_HIGH":
            "APSIM_A150_T_FFB_HA_YR",
    }
)

frozen = base_keep.merge(
    sens_yield,
    on="UNIT_ID",
    how="left",
    validate="one_to_one",
)

# ------------------------------------------------------------
# 5. DERIVED SENSITIVITY METRICS
# ------------------------------------------------------------

frozen["APSIM_A130_DELTA_T_HA_YR"] = (
    frozen["APSIM_A130_T_FFB_HA_YR"]
    - frozen["APSIM_A143_BASE_T_FFB_HA_YR"]
)

frozen["APSIM_A150_DELTA_T_HA_YR"] = (
    frozen["APSIM_A150_T_FFB_HA_YR"]
    - frozen["APSIM_A143_BASE_T_FFB_HA_YR"]
)

scenario_cols = [
    "APSIM_A130_T_FFB_HA_YR",
    "APSIM_A143_BASE_T_FFB_HA_YR",
    "APSIM_A150_T_FFB_HA_YR",
]

frozen["APSIM_POP_RESPONSE_RANGE_T_HA_YR"] = (
    frozen[scenario_cols].max(axis=1)
    - frozen[scenario_cols].min(axis=1)
)

frozen["APSIM_POP_RESPONSE_RANGE_PCT"] = np.where(
    frozen["APSIM_A143_BASE_T_FFB_HA_YR"] > 1.0,

    100.0
    * frozen["APSIM_POP_RESPONSE_RANGE_T_HA_YR"]
    / frozen["APSIM_A143_BASE_T_FFB_HA_YR"],

    np.nan,
)

# ------------------------------------------------------------
# 6. CROSS-CHECK BASELINE STORED INSIDE SENSITIVITY RUNS
# ------------------------------------------------------------

sens_base = (
    sens.groupby("UNIT_ID")["BASE_YIELD_T_FFB_HA_YR"]
    .first()
)

canonical_base = (
    frozen.set_index("UNIT_ID")
    ["APSIM_A143_BASE_T_FFB_HA_YR"]
)

common_ids = canonical_base.index.intersection(
    sens_base.index
)

baseline_diff = (
    canonical_base.loc[common_ids]
    - sens_base.loc[common_ids]
).abs()

max_baseline_diff = float(baseline_diff.max())

# ------------------------------------------------------------
# 7. FINAL QC
# ------------------------------------------------------------

numeric_cols = [
    "APSIM_A130_T_FFB_HA_YR",
    "APSIM_A143_BASE_T_FFB_HA_YR",
    "APSIM_A150_T_FFB_HA_YR",
    "APSIM_POP_RESPONSE_RANGE_T_HA_YR",
]

checks = {
    "frozen_rows_48":
        len(frozen) == 48,

    "unique_asu_48":
        frozen["UNIT_ID"].nunique() == 48,

    "climate_pairs_12":
        frozen["CLIMATE_PAIR_ID"].nunique() == 12,

    "baseline_population_143":
        bool(
            (
                frozen["POPULATION_PALM_HA"] == 143
            ).all()
        ),

    "all_three_yields_finite":
        bool(
            np.isfinite(
                frozen[numeric_cols]
                .apply(pd.to_numeric, errors="coerce")
                .to_numpy()
            ).all()
        ),

    "age_match_within_005yr":
        bool(
            (
                frozen["age_match_abs_diff_yr"]
                <= 0.05
            ).all()
        ),

    "sensitivity_baseline_matches_canonical":
        max_baseline_diff <= TOL,

    "no_fixed_endcrop_policy":
        bool(
            frozen["ENDCROP_POLICY"]
            .astype(str)
            .str.contains(
                "NO_FIXED_ENDCROP",
                case=False,
                regex=False,
            )
            .all()
        ),

    "soil_policy_reference_only":
        bool(
            frozen["SOIL_POLICY"]
            .astype(str)
            .str.contains(
                "REFERENCE",
                case=False,
                regex=False,
            )
            .all()
        ),
}

status = (
    "PASS_FROZEN"
    if all(checks.values())
    else "REVIEW"
)

# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

y = frozen["APSIM_A143_BASE_T_FFB_HA_YR"]

response_pct = (
    frozen["APSIM_POP_RESPONSE_RANGE_PCT"]
    .dropna()
)

summary = {
    "baseline_min_t_ha_yr":
        float(y.min()),

    "baseline_median_t_ha_yr":
        float(y.median()),

    "baseline_mean_t_ha_yr":
        float(y.mean()),

    "baseline_max_t_ha_yr":
        float(y.max()),

    "population_response_range_median_t_ha_yr":
        float(
            frozen[
                "APSIM_POP_RESPONSE_RANGE_T_HA_YR"
            ].median()
        ),

    "population_response_range_p95_t_ha_yr":
        float(
            frozen[
                "APSIM_POP_RESPONSE_RANGE_T_HA_YR"
            ].quantile(0.95)
        ),

    "population_response_range_median_pct_base_gt_1":
        float(response_pct.median())
        if len(response_pct)
        else None,
}

# ------------------------------------------------------------
# 9. FREEZE METADATA
# ------------------------------------------------------------

frozen["APSIM_BENCHMARK_STATUS"] = (
    "REPRESENTATIVE_PROCESS_BENCHMARK"
)

frozen["APSIM_LOCAL_YIELD_VALIDATED"] = False

frozen["APSIM_CANONICAL_OUTPUT"] = (
    "SQLite AnnualOutput / "
    "Calculations.Script.AnnualYield"
)

frozen["APSIM_CANONICAL_UNIT"] = (
    "t fresh FFB/ha/year"
)

frozen["APSIM_VERSION"] = (
    "2026.8.8099.0"
)

frozen["APSIM_FREEZE_VERSION"] = (
    "40D05_V1"
)

# ------------------------------------------------------------
# 10. SAVE
# ------------------------------------------------------------

frozen.to_csv(
    OUT_CSV,
    index=False,
)

qc = {
    "gate":
        "40D05_APSIM_BENCHMARK_FINAL_QC_FREEZE",

    "status":
        status,

    "apsim_version":
        "2026.8.8099.0",

    "representative_asus":
        int(len(frozen)),

    "climate_pairs":
        int(
            frozen["CLIMATE_PAIR_ID"].nunique()
        ),

    "canonical_baseline_population_palm_ha":
        143,

    "population_sensitivity":
        {
            "low": 130,
            "baseline": 143,
            "high": 150,
        },

    "canonical_output_source":
        "SQLite .db / AnnualOutput",

    "canonical_output_field":
        "Calculations.Script.AnnualYield",

    "canonical_unit":
        "t fresh FFB/ha/year",

    "cultivar_policy":
        "Dami / official APSIM reference",

    "soil_policy":
        (
            "Official APSIM OilPalm reference soil; "
            "not locally calibrated to Siak."
        ),

    "management_policy":
        (
            "Official reference management except "
            "example-specific fixed 2007 EndCrop disabled."
        ),

    "weather_policy":
        (
            "CHIRPS V2 rainfall + ERA5-Land "
            "radiation/Tmin/Tmax."
        ),

    "age_policy":
        (
            "Output sampled at OilPalm.Age nearest "
            "ASU AGE_PROXY_YR."
        ),

    "max_sensitivity_vs_canonical_baseline_diff":
        max_baseline_diff,

    "summary":
        summary,

    "checks":
        checks,

    "interpretation_limits": [
        "Representative process/scenario benchmark only.",
        "Not actual observed FFB yield.",
        "Not locally calibrated against plantation yield records.",
        "Population sensitivity is model/scenario response, not empirical uncertainty.",
        "Reference APSIM soil is not a measured Siak soil profile.",
        "Do not blindly average APSIM output with PALMSIM.",
    ],

    "next":
        "40E01 PALMSIM vs APSIM representative process-model comparison.",
}

OUT_JSON.write_text(
    json.dumps(
        qc,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# PRINT
# ------------------------------------------------------------

print("\n====================================================")
print("40D05 FINISHED")
print("====================================================")

print("Status                  :", status)
print("Representative ASUs     :", len(frozen))
print("Climate pairs           :", frozen["CLIMATE_PAIR_ID"].nunique())

print("\nAPSIM baseline:")
print("Min                      :", summary["baseline_min_t_ha_yr"])
print("Median                   :", summary["baseline_median_t_ha_yr"])
print("Mean                     :", summary["baseline_mean_t_ha_yr"])
print("Max                      :", summary["baseline_max_t_ha_yr"])

print("\nPopulation sensitivity:")
print(
    "Median range            :",
    summary[
        "population_response_range_median_t_ha_yr"
    ],
    "t FFB/ha/yr",
)

print(
    "P95 range               :",
    summary[
        "population_response_range_p95_t_ha_yr"
    ],
    "t FFB/ha/yr",
)

print(
    "Median range % (>1 base):",
    summary[
        "population_response_range_median_pct_base_gt_1"
    ],
    "%",
)

print(
    "\nMax baseline cross-check diff:",
    max_baseline_diff,
)

print("\nQC:")
for k, v in checks.items():
    print(f"{k:48s}: {v}")

print("\nFrozen CSV :", OUT_CSV)
print("QC JSON    :", OUT_JSON)

if status == "PASS_FROZEN":
    print(
        "Next       : 40E01 PALMSIM vs APSIM "
        "representative process-model comparison."
    )
else:
    print(
        "Next       : Review failed freeze QC "
        "before 40E."
    )

print("====================================================")

40D05 — APSIM BENCHMARK FINAL QC / FREEZE
Baseline rows    : 48
Sensitivity rows : 96

40D05 FINISHED
Status                  : PASS_FROZEN
Representative ASUs     : 48
Climate pairs           : 12

APSIM baseline:
Min                      : 0.1267306766622727
Median                   : 19.297824782811592
Mean                     : 17.435889214450143
Max                      : 33.850461798272846

Population sensitivity:
Median range            : 1.128659799931425 t FFB/ha/yr
P95 range               : 3.108142575912674 t FFB/ha/yr
Median range % (>1 base): 9.352432070742974 %

Max baseline cross-check diff: 3.552713678800501e-15

QC:
frozen_rows_48                                  : True
unique_asu_48                                   : True
climate_pairs_12                                : True
baseline_population_143                         : True
all_three_yields_finite                         : True
age_match_within_005yr                          : True
sensitivity_baseline_matches_

In [ ]:
# ============================================================
# OUTPUT 4 / 40E01A
# PALMSIM vs APSIM — CORE PROCESS-MODEL COMPARISON
# SIAK_AOI01
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

# PALMSIM full baseline: 654 eligible ASUs
PALMSIM_BASE = WORKDIR / (
    "40C04A_DIRECT_P00_BASELINE_RESULTS_SIAK_AOI01_V1.csv"
)

# Frozen 48-ASU representative set
PALMSIM_SAMPLE = WORKDIR / (
    "40C04B_REPRESENTATIVE_SENSITIVITY_SAMPLE_SIAK_AOI01_V1.csv"
)

# Frozen APSIM benchmark
APSIM = WORKDIR / (
    "40D05_APSIM_REPRESENTATIVE_BENCHMARK_FROZEN_SIAK_AOI01_V1.csv"
)

OUT_CSV = WORKDIR / (
    "40E01A_PALMSIM_APSIM_CORE_COMPARISON_SIAK_AOI01_V1.csv"
)

OUT_JSON = WORKDIR / (
    "40E01A_PALMSIM_APSIM_CORE_COMPARISON_QC_SIAK_AOI01_V1.json"
)

print("====================================================")
print("40E01A — PALMSIM vs APSIM CORE COMPARISON")
print("====================================================")

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

for p in [PALMSIM_BASE, PALMSIM_SAMPLE, APSIM]:
    if not p.exists():
        raise FileNotFoundError(p)

palm = pd.read_csv(PALMSIM_BASE)
sample = pd.read_csv(PALMSIM_SAMPLE)
apsim = pd.read_csv(APSIM)

print("PALMSIM baseline rows :", len(palm))
print("Representative sample:", len(sample))
print("APSIM frozen rows     :", len(apsim))

# ------------------------------------------------------------
# 2. RESOLVE PALMSIM CANONICAL YIELD COLUMN
# ------------------------------------------------------------

preferred = [
    "TRAILING_1Y_MEAN_T_HA_YR",
    "POTENTIAL_T_HA_YR",
    "BASELINE_T_HA_YR",
    "FFB_T_HA_YR",
]

palm_yield_col = None

for c in preferred:
    if c in palm.columns:
        palm_yield_col = c
        break

# Fallback: find a likely trailing-1y output.
if palm_yield_col is None:
    candidates = [
        c for c in palm.columns
        if (
            "T_HA_YR" in c.upper()
            and (
                "TRAIL" in c.upper()
                or "POTENTIAL" in c.upper()
                or "BASE" in c.upper()
            )
        )
    ]

    if len(candidates) == 1:
        palm_yield_col = candidates[0]

if palm_yield_col is None:
    print("\nPALMSIM columns:")
    print(list(palm.columns))
    raise ValueError(
        "Could not uniquely identify canonical PALMSIM yield column."
    )

print("\nPALMSIM canonical yield:", palm_yield_col)

# ------------------------------------------------------------
# 3. VERIFY REPRESENTATIVE SET
# ------------------------------------------------------------

if "UNIT_ID" not in palm.columns:
    raise ValueError("UNIT_ID missing from PALMSIM baseline.")

if "UNIT_ID" not in sample.columns:
    raise ValueError("UNIT_ID missing from representative sample.")

if "UNIT_ID" not in apsim.columns:
    raise ValueError("UNIT_ID missing from APSIM frozen table.")

sample_ids = set(sample["UNIT_ID"].astype(str))
apsim_ids = set(apsim["UNIT_ID"].astype(str))

if sample_ids != apsim_ids:
    raise ValueError(
        "PALMSIM representative sample and APSIM frozen ASU set differ."
    )

if len(sample_ids) != 48:
    raise ValueError(
        f"Expected 48 representative ASUs; got {len(sample_ids)}."
    )

# ------------------------------------------------------------
# 4. PREPARE PALMSIM
# ------------------------------------------------------------

palm48 = palm.loc[
    palm["UNIT_ID"].astype(str).isin(sample_ids),
    ["UNIT_ID", palm_yield_col]
].copy()

if len(palm48) != 48:
    raise ValueError(
        f"Expected 48 PALMSIM rows; found {len(palm48)}."
    )

if palm48["UNIT_ID"].duplicated().any():
    raise ValueError("Duplicate PALMSIM UNIT_ID.")

palm48 = palm48.rename(
    columns={
        palm_yield_col:
            "PALMSIM_BASE_T_FFB_HA_YR"
    }
)

# ------------------------------------------------------------
# 5. PREPARE APSIM
# ------------------------------------------------------------

apsim_keep = apsim[
    [
        "UNIT_ID",
        "AGE_PROXY_YR",
        "CLIMATE_PAIR_ID",
        "APSIM_A143_BASE_T_FFB_HA_YR",
        "APSIM_POP_RESPONSE_RANGE_T_HA_YR",
        "APSIM_POP_RESPONSE_RANGE_PCT",
    ]
].copy()

# ------------------------------------------------------------
# 6. JOIN
# ------------------------------------------------------------

comp = apsim_keep.merge(
    palm48,
    on="UNIT_ID",
    how="left",
    validate="one_to_one",
)

# ------------------------------------------------------------
# 7. CORE COMPARISON METRICS PER ASU
# ------------------------------------------------------------

A = pd.to_numeric(
    comp["APSIM_A143_BASE_T_FFB_HA_YR"],
    errors="coerce"
)

P = pd.to_numeric(
    comp["PALMSIM_BASE_T_FFB_HA_YR"],
    errors="coerce"
)

comp["MODEL_DELTA_APSIM_MINUS_PALMSIM_T_HA_YR"] = A - P

comp["MODEL_ABS_DIFF_T_HA_YR"] = (A - P).abs()

# Symmetric percentage difference:
# 0 = same; avoids treating one model as "truth".
denom = A.abs() + P.abs()

comp["MODEL_SYMMETRIC_DIFF_PCT"] = np.where(
    denom > 0,
    200.0 * (A - P).abs() / denom,
    0.0
)

# Ratio is diagnostic only; suppress where PALMSIM nearly zero.
comp["APSIM_TO_PALMSIM_RATIO"] = np.where(
    P > 0.1,
    A / P,
    np.nan
)

# Percentile ranks for ranking comparison.
comp["PALMSIM_PERCENTILE_RANK"] = (
    P.rank(method="average", pct=True)
)

comp["APSIM_PERCENTILE_RANK"] = (
    A.rank(method="average", pct=True)
)

comp["MODEL_RANK_ABS_DIFF"] = (
    comp["APSIM_PERCENTILE_RANK"]
    - comp["PALMSIM_PERCENTILE_RANK"]
).abs()

# ------------------------------------------------------------
# 8. GLOBAL MODEL-TO-MODEL METRICS
# ------------------------------------------------------------

finite = np.isfinite(A) & np.isfinite(P)

if not finite.all():
    raise ValueError("Non-finite PALMSIM/APSIM yield values detected.")

delta = A - P

mae = float(np.mean(np.abs(delta)))
rmse = float(np.sqrt(np.mean(delta ** 2)))
bias_mean = float(np.mean(delta))
bias_median = float(np.median(delta))

pearson = float(
    pd.Series(P).corr(
        pd.Series(A),
        method="pearson"
    )
)

spearman = float(
    pd.Series(P).corr(
        pd.Series(A),
        method="spearman"
    )
)

symdiff = comp["MODEL_SYMMETRIC_DIFF_PCT"]

rankdiff = comp["MODEL_RANK_ABS_DIFF"]

ratio = comp["APSIM_TO_PALMSIM_RATIO"].dropna()

# ------------------------------------------------------------
# 9. QC
# ------------------------------------------------------------

checks = {
    "comparison_rows_48":
        len(comp) == 48,

    "unique_asu_48":
        comp["UNIT_ID"].nunique() == 48,

    "same_representative_set":
        set(comp["UNIT_ID"].astype(str)) == sample_ids,

    "climate_pairs_12":
        comp["CLIMATE_PAIR_ID"].nunique() == 12,

    "all_palmsim_finite":
        bool(np.isfinite(P).all()),

    "all_apsim_finite":
        bool(np.isfinite(A).all()),

    "all_abs_diff_finite":
        bool(
            np.isfinite(
                comp["MODEL_ABS_DIFF_T_HA_YR"]
            ).all()
        ),
}

# NOTE:
# Correlation/bias are NOT used as pass/fail gates.
# Model disagreement itself is a scientific result.
status = (
    "PASS_COMPARISON_COMPLETE"
    if all(checks.values())
    else "REVIEW"
)

# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

summary = {
    "palmsim_min_t_ha_yr":
        float(P.min()),

    "palmsim_median_t_ha_yr":
        float(P.median()),

    "palmsim_mean_t_ha_yr":
        float(P.mean()),

    "palmsim_max_t_ha_yr":
        float(P.max()),

    "apsim_min_t_ha_yr":
        float(A.min()),

    "apsim_median_t_ha_yr":
        float(A.median()),

    "apsim_mean_t_ha_yr":
        float(A.mean()),

    "apsim_max_t_ha_yr":
        float(A.max()),

    "mean_bias_apsim_minus_palmsim_t_ha_yr":
        bias_mean,

    "median_bias_apsim_minus_palmsim_t_ha_yr":
        bias_median,

    "mae_t_ha_yr":
        mae,

    "rmse_t_ha_yr":
        rmse,

    "pearson_r":
        pearson,

    "spearman_rho":
        spearman,

    "median_symmetric_difference_pct":
        float(symdiff.median()),

    "p95_symmetric_difference_pct":
        float(symdiff.quantile(0.95)),

    "median_absolute_rank_difference":
        float(rankdiff.median()),

    "p95_absolute_rank_difference":
        float(rankdiff.quantile(0.95)),

    "median_apsim_to_palmsim_ratio":
        float(ratio.median())
        if len(ratio)
        else None,
}

# ------------------------------------------------------------
# 11. METADATA / SAVE
# ------------------------------------------------------------

comp["COMPARISON_STATUS"] = (
    "PROCESS_MODEL_DIAGNOSTIC_ONLY"
)

comp["LOCAL_YIELD_VALIDATION"] = False

comp["MODEL_AGGREGATION_APPLIED"] = False

comp["COMPARISON_VERSION"] = "40E01A_V1"

comp.to_csv(
    OUT_CSV,
    index=False,
)

qc = {
    "gate":
        "40E01A_PALMSIM_APSIM_CORE_PROCESS_MODEL_COMPARISON",

    "status":
        status,

    "representative_asus":
        int(len(comp)),

    "climate_pairs":
        int(comp["CLIMATE_PAIR_ID"].nunique()),

    "palmsim_source":
        PALMSIM_BASE.name,

    "palmsim_yield_field":
        palm_yield_col,

    "apsim_source":
        APSIM.name,

    "apsim_yield_field":
        "APSIM_A143_BASE_T_FFB_HA_YR",

    "unit":
        "t fresh FFB/ha/year",

    "summary":
        summary,

    "checks":
        checks,

    "interpretation": [
        "This is model-to-model comparison, not validation against observed FFB.",
        "Neither PALMSIM nor APSIM is treated as ground truth.",
        "Correlation measures co-variation/ranking, not model accuracy.",
        "Bias means APSIM minus PALMSIM, not prediction error.",
        "No PALMSIM/APSIM averaging is performed.",
        "Differences will be diagnosed by age and climate pair in the next step.",
    ],

    "next":
        "40E01B discrepancy diagnostics by age, yield regime, and climate pair.",
}

OUT_JSON.write_text(
    json.dumps(
        qc,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 12. PRINT
# ------------------------------------------------------------

print("\n====================================================")
print("40E01A FINISHED")
print("====================================================")

print("Status            :", status)
print("ASUs              :", len(comp))
print("Climate pairs     :", comp["CLIMATE_PAIR_ID"].nunique())

print("\nPALMSIM:")
print("Median            :", summary["palmsim_median_t_ha_yr"])
print("Mean              :", summary["palmsim_mean_t_ha_yr"])

print("\nAPSIM:")
print("Median            :", summary["apsim_median_t_ha_yr"])
print("Mean              :", summary["apsim_mean_t_ha_yr"])

print("\nModel-to-model:")
print(
    "Mean bias A-P     :",
    summary[
        "mean_bias_apsim_minus_palmsim_t_ha_yr"
    ]
)

print(
    "Median bias A-P   :",
    summary[
        "median_bias_apsim_minus_palmsim_t_ha_yr"
    ]
)

print("MAE               :", mae)
print("RMSE              :", rmse)
print("Pearson r         :", pearson)
print("Spearman rho      :", spearman)

print(
    "Median sym diff % :",
    summary[
        "median_symmetric_difference_pct"
    ]
)

print(
    "Median rank diff  :",
    summary[
        "median_absolute_rank_difference"
    ]
)

print("\nQC:")
for k, v in checks.items():
    print(f"{k:42s}: {v}")

print("\nComparison CSV :", OUT_CSV)
print("QC JSON        :", OUT_JSON)

if status == "PASS_COMPARISON_COMPLETE":
    print(
        "Next           : 40E01B discrepancy diagnostics "
        "by age and climate pair."
    )
else:
    print(
        "Next           : Review comparison integrity "
        "before diagnostic interpretation."
    )

print("====================================================")

40E01A — PALMSIM vs APSIM CORE COMPARISON
PALMSIM baseline rows : 654
Representative sample: 48
APSIM frozen rows     : 48

PALMSIM canonical yield: TRAILING_1Y_MEAN_T_HA_YR


In [1]:
print("KERNEL OK")

KERNEL OK


In [ ]:
# ============================================================
# OUTPUT 4 / 40E01A-V2
# PALMSIM vs APSIM — LIGHTWEIGHT CORE COMPARISON
# SIAK_AOI01
#
# Pure NumPy + Pandas
# No scipy
# No APSIM rerun
# No PALMSIM rerun
# ============================================================

from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. CLEAN MEMORY
# ------------------------------------------------------------

gc.collect()

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

PALMSIM_BASE = WORKDIR / (
    "40C04A_DIRECT_P00_BASELINE_RESULTS_SIAK_AOI01_V1.csv"
)

REP_SAMPLE = WORKDIR / (
    "40C04B_REPRESENTATIVE_SENSITIVITY_SAMPLE_SIAK_AOI01_V1.csv"
)

APSIM_FROZEN = WORKDIR / (
    "40D05_APSIM_REPRESENTATIVE_BENCHMARK_FROZEN_SIAK_AOI01_V1.csv"
)

OUT_CSV = WORKDIR / (
    "40E01A_V2_PALMSIM_APSIM_CORE_COMPARISON_SIAK_AOI01.csv"
)

OUT_JSON = WORKDIR / (
    "40E01A_V2_PALMSIM_APSIM_CORE_COMPARISON_QC_SIAK_AOI01.json"
)

print("====================================================")
print("40E01A-V2 — PALMSIM vs APSIM CORE COMPARISON")
print("LIGHTWEIGHT / NO SCIPY")
print("====================================================")

# ------------------------------------------------------------
# 1. PREFLIGHT
# ------------------------------------------------------------

for p in [PALMSIM_BASE, REP_SAMPLE, APSIM_FROZEN]:
    if not p.exists():
        raise FileNotFoundError(p)

# Load only what is required.
sample = pd.read_csv(
    REP_SAMPLE,
    usecols=["UNIT_ID"]
)

apsim = pd.read_csv(
    APSIM_FROZEN,
    usecols=[
        "UNIT_ID",
        "AGE_PROXY_YR",
        "CLIMATE_PAIR_ID",
        "APSIM_A143_BASE_T_FFB_HA_YR",
        "APSIM_POP_RESPONSE_RANGE_T_HA_YR",
        "APSIM_POP_RESPONSE_RANGE_PCT",
    ]
)

# PALMSIM baseline schema first.
palm_header = pd.read_csv(
    PALMSIM_BASE,
    nrows=0
)

print("Representative sample :", len(sample))
print("APSIM frozen           :", len(apsim))

# ------------------------------------------------------------
# 2. RESOLVE PALMSIM CANONICAL YIELD FIELD
# ------------------------------------------------------------

preferred = [
    "TRAILING_1Y_MEAN_T_HA_YR",
    "POTENTIAL_T_HA_YR",
    "BASELINE_T_HA_YR",
    "FFB_T_HA_YR",
]

palm_yield_col = next(
    (c for c in preferred if c in palm_header.columns),
    None
)

if palm_yield_col is None:

    candidates = [
        c for c in palm_header.columns
        if (
            "T_HA_YR" in c.upper()
            and (
                "TRAIL" in c.upper()
                or "POTENTIAL" in c.upper()
                or "BASE" in c.upper()
            )
        )
    ]

    if len(candidates) != 1:
        print("\nPALMSIM columns:")
        print(list(palm_header.columns))

        raise ValueError(
            "PALMSIM canonical yield field could not "
            "be uniquely identified."
        )

    palm_yield_col = candidates[0]

print("PALMSIM yield field    :", palm_yield_col)

# Now load only UNIT_ID + required yield.
palm = pd.read_csv(
    PALMSIM_BASE,
    usecols=[
        "UNIT_ID",
        palm_yield_col,
    ]
)

# ------------------------------------------------------------
# 3. REPRESENTATIVE SET QC
# ------------------------------------------------------------

sample_ids = set(
    sample["UNIT_ID"].astype(str)
)

apsim_ids = set(
    apsim["UNIT_ID"].astype(str)
)

checks = {}

checks["representative_sample_48"] = (
    len(sample) == 48
    and sample["UNIT_ID"].nunique() == 48
)

checks["apsim_frozen_48"] = (
    len(apsim) == 48
    and apsim["UNIT_ID"].nunique() == 48
)

checks["same_representative_set"] = (
    sample_ids == apsim_ids
)

if not all(checks.values()):
    raise ValueError(
        f"Representative set integrity failed: {checks}"
    )

# Keep only the exact 48 PALMSIM ASUs.
palm48 = palm.loc[
    palm["UNIT_ID"].astype(str).isin(sample_ids)
].copy()

if len(palm48) != 48:
    raise ValueError(
        f"Expected 48 PALMSIM representative rows; "
        f"found {len(palm48)}."
    )

if palm48["UNIT_ID"].duplicated().any():
    raise ValueError(
        "Duplicate UNIT_ID in PALMSIM representative baseline."
    )

palm48 = palm48.rename(
    columns={
        palm_yield_col:
            "PALMSIM_BASE_T_FFB_HA_YR"
    }
)

# Free full PALMSIM table.
del palm
gc.collect()

# ------------------------------------------------------------
# 4. JOIN
# ------------------------------------------------------------

comp = apsim.merge(
    palm48,
    on="UNIT_ID",
    how="inner",
    validate="one_to_one",
)

if len(comp) != 48:
    raise ValueError(
        f"Comparison join returned {len(comp)} rows, expected 48."
    )

A = pd.to_numeric(
    comp["APSIM_A143_BASE_T_FFB_HA_YR"],
    errors="coerce"
).to_numpy(dtype=float)

P = pd.to_numeric(
    comp["PALMSIM_BASE_T_FFB_HA_YR"],
    errors="coerce"
).to_numpy(dtype=float)

if not (
    np.isfinite(A).all()
    and np.isfinite(P).all()
):
    raise ValueError(
        "Non-finite APSIM/PALMSIM baseline values found."
    )

# ------------------------------------------------------------
# 5. PER-ASU DIFFERENCE METRICS
# ------------------------------------------------------------

delta = A - P

comp[
    "MODEL_DELTA_APSIM_MINUS_PALMSIM_T_HA_YR"
] = delta

comp[
    "MODEL_ABS_DIFF_T_HA_YR"
] = np.abs(delta)

denom = np.abs(A) + np.abs(P)

comp[
    "MODEL_SYMMETRIC_DIFF_PCT"
] = np.where(
    denom > 0,
    200.0 * np.abs(delta) / denom,
    0.0
)

# Ratio only where PALMSIM isn't effectively zero.
comp[
    "APSIM_TO_PALMSIM_RATIO"
] = np.where(
    P > 0.1,
    A / P,
    np.nan
)

# ------------------------------------------------------------
# 6. RANKS — PANDAS ONLY
# ------------------------------------------------------------

p_rank = pd.Series(P).rank(
    method="average",
    pct=True
).to_numpy()

a_rank = pd.Series(A).rank(
    method="average",
    pct=True
).to_numpy()

comp[
    "PALMSIM_PERCENTILE_RANK"
] = p_rank

comp[
    "APSIM_PERCENTILE_RANK"
] = a_rank

comp[
    "MODEL_RANK_ABS_DIFF"
] = np.abs(
    a_rank - p_rank
)

# ------------------------------------------------------------
# 7. CORRELATION — PURE NUMPY
# ------------------------------------------------------------

def safe_corr(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if (
        len(x) < 2
        or np.std(x) == 0
        or np.std(y) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(x, y)[0, 1]
    )


pearson_r = safe_corr(P, A)

# Spearman = Pearson correlation of ranks.
spearman_rho = safe_corr(
    p_rank,
    a_rank
)

# ------------------------------------------------------------
# 8. GLOBAL DIAGNOSTICS
# ------------------------------------------------------------

mae = float(
    np.mean(
        np.abs(delta)
    )
)

rmse = float(
    np.sqrt(
        np.mean(
            delta ** 2
        )
    )
)

bias_mean = float(
    np.mean(delta)
)

bias_median = float(
    np.median(delta)
)

symdiff = comp[
    "MODEL_SYMMETRIC_DIFF_PCT"
]

rankdiff = comp[
    "MODEL_RANK_ABS_DIFF"
]

ratio = comp[
    "APSIM_TO_PALMSIM_RATIO"
].dropna()

# ------------------------------------------------------------
# 9. FINAL INTEGRITY QC
# ------------------------------------------------------------

checks.update({

    "comparison_rows_48":
        len(comp) == 48,

    "unique_comparison_asu_48":
        comp["UNIT_ID"].nunique() == 48,

    "climate_pairs_12":
        comp["CLIMATE_PAIR_ID"].nunique() == 12,

    "all_palmsim_finite":
        bool(
            np.isfinite(P).all()
        ),

    "all_apsim_finite":
        bool(
            np.isfinite(A).all()
        ),

    "all_abs_diff_finite":
        bool(
            np.isfinite(
                comp[
                    "MODEL_ABS_DIFF_T_HA_YR"
                ].to_numpy(dtype=float)
            ).all()
        ),
})

status = (
    "PASS_COMPARISON_COMPLETE"
    if all(checks.values())
    else "REVIEW"
)

# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

summary = {

    "palmsim_min_t_ha_yr":
        float(np.min(P)),

    "palmsim_median_t_ha_yr":
        float(np.median(P)),

    "palmsim_mean_t_ha_yr":
        float(np.mean(P)),

    "palmsim_max_t_ha_yr":
        float(np.max(P)),

    "apsim_min_t_ha_yr":
        float(np.min(A)),

    "apsim_median_t_ha_yr":
        float(np.median(A)),

    "apsim_mean_t_ha_yr":
        float(np.mean(A)),

    "apsim_max_t_ha_yr":
        float(np.max(A)),

    "mean_bias_apsim_minus_palmsim_t_ha_yr":
        bias_mean,

    "median_bias_apsim_minus_palmsim_t_ha_yr":
        bias_median,

    "mae_t_ha_yr":
        mae,

    "rmse_t_ha_yr":
        rmse,

    "pearson_r":
        pearson_r,

    "spearman_rho":
        spearman_rho,

    "median_symmetric_difference_pct":
        float(
            symdiff.median()
        ),

    "p95_symmetric_difference_pct":
        float(
            symdiff.quantile(0.95)
        ),

    "median_absolute_rank_difference":
        float(
            rankdiff.median()
        ),

    "p95_absolute_rank_difference":
        float(
            rankdiff.quantile(0.95)
        ),

    "median_apsim_to_palmsim_ratio":
        (
            float(ratio.median())
            if len(ratio)
            else None
        ),
}

# ------------------------------------------------------------
# 11. FREEZE COMPARISON TABLE
# ------------------------------------------------------------

comp[
    "COMPARISON_STATUS"
] = "PROCESS_MODEL_DIAGNOSTIC_ONLY"

comp[
    "LOCAL_YIELD_VALIDATION"
] = False

comp[
    "MODEL_AGGREGATION_APPLIED"
] = False

comp[
    "COMPARISON_VERSION"
] = "40E01A_V2"

comp.to_csv(
    OUT_CSV,
    index=False
)

qc = {

    "gate":
        "40E01A_V2_PALMSIM_APSIM_CORE_COMPARISON",

    "status":
        status,

    "representative_asus":
        int(len(comp)),

    "climate_pairs":
        int(
            comp[
                "CLIMATE_PAIR_ID"
            ].nunique()
        ),

    "palmsim_yield_field":
        palm_yield_col,

    "apsim_yield_field":
        "APSIM_A143_BASE_T_FFB_HA_YR",

    "unit":
        "t fresh FFB/ha/year",

    "summary":
        summary,

    "checks":
        checks,

    "interpretation_limits": [
        "Model-to-model comparison only.",
        "Neither model is ground truth.",
        "Not validation against observed plantation FFB.",
        "Bias is APSIM minus PALMSIM, not prediction error.",
        "Correlation describes co-variation/ranking, not accuracy.",
        "No PALMSIM/APSIM averaging is applied.",
    ],

    "next":
        "40E01B discrepancy diagnostics by age and climate pair.",
}

OUT_JSON.write_text(
    json.dumps(
        qc,
        indent=2
    ),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 12. PRINT
# ------------------------------------------------------------

print("\n====================================================")
print("40E01A-V2 FINISHED")
print("====================================================")

print("Status            :", status)
print("ASUs              :", len(comp))
print(
    "Climate pairs     :",
    comp["CLIMATE_PAIR_ID"].nunique()
)

print("\nPALMSIM")
print(
    "Min               :",
    summary["palmsim_min_t_ha_yr"]
)
print(
    "Median            :",
    summary["palmsim_median_t_ha_yr"]
)
print(
    "Mean              :",
    summary["palmsim_mean_t_ha_yr"]
)
print(
    "Max               :",
    summary["palmsim_max_t_ha_yr"]
)

print("\nAPSIM")
print(
    "Min               :",
    summary["apsim_min_t_ha_yr"]
)
print(
    "Median            :",
    summary["apsim_median_t_ha_yr"]
)
print(
    "Mean              :",
    summary["apsim_mean_t_ha_yr"]
)
print(
    "Max               :",
    summary["apsim_max_t_ha_yr"]
)

print("\nMODEL-TO-MODEL")
print(
    "Mean bias A-P     :",
    bias_mean
)
print(
    "Median bias A-P   :",
    bias_median
)
print(
    "MAE               :",
    mae
)
print(
    "RMSE              :",
    rmse
)
print(
    "Pearson r         :",
    pearson_r
)
print(
    "Spearman rho      :",
    spearman_rho
)
print(
    "Median sym diff % :",
    summary[
        "median_symmetric_difference_pct"
    ]
)
print(
    "Median rank diff  :",
    summary[
        "median_absolute_rank_difference"
    ]
)

print("\nQC:")
for k, v in checks.items():
    print(
        f"{k:42s}: {v}"
    )

print("\nComparison CSV :", OUT_CSV)
print("QC JSON        :", OUT_JSON)

if status == "PASS_COMPARISON_COMPLETE":
    print(
        "Next           : 40E01B discrepancy diagnostics "
        "by age and climate pair."
    )
else:
    print(
        "Next           : Review integrity before 40E01B."
    )

print("====================================================")

# Final cleanup of objects not needed after save.
del sample, apsim, palm48, palm_header
gc.collect()

40E01A-V2 — PALMSIM vs APSIM CORE COMPARISON
LIGHTWEIGHT / NO SCIPY
Representative sample : 48
APSIM frozen           : 48
PALMSIM yield field    : TRAILING_1Y_MEAN_T_HA_YR


C:\Users\user\AppData\Local\Temp\ipykernel_9140\2037830142.py:263: RuntimeWarning: divide by zero encountered in divide
  A / P,


In [1]:
print("KERNEL OK")

KERNEL OK


In [2]:
# ============================================================
# OUTPUT 4 / 40E01A-V3
# PALMSIM vs APSIM — PURE PYTHON CORE COMPARISON
# SIAK_AOI01
#
# NO numpy
# NO pandas
# NO scipy
# NO model rerun
# ============================================================

from pathlib import Path
import csv
import json
import math
import statistics

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

PALMSIM_BASE = WORKDIR / (
    "40C04A_DIRECT_P00_BASELINE_RESULTS_SIAK_AOI01_V1.csv"
)

REP_SAMPLE = WORKDIR / (
    "40C04B_REPRESENTATIVE_SENSITIVITY_SAMPLE_SIAK_AOI01_V1.csv"
)

APSIM_FROZEN = WORKDIR / (
    "40D05_APSIM_REPRESENTATIVE_BENCHMARK_FROZEN_SIAK_AOI01_V1.csv"
)

OUT_CSV = WORKDIR / (
    "40E01A_V3_PALMSIM_APSIM_CORE_COMPARISON_SIAK_AOI01.csv"
)

OUT_JSON = WORKDIR / (
    "40E01A_V3_PALMSIM_APSIM_CORE_COMPARISON_QC_SIAK_AOI01.json"
)

print("====================================================")
print("40E01A-V3 — PALMSIM vs APSIM CORE COMPARISON")
print("PURE PYTHON / NO NUMPY / NO PANDAS / NO SCIPY")
print("====================================================")


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def read_csv_dict(path):
    with open(
        path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:
        return list(csv.DictReader(f))


def to_float(value, name="value"):
    try:
        x = float(value)
    except Exception:
        raise ValueError(
            f"Cannot convert {name}={value!r} to float."
        )

    if not math.isfinite(x):
        raise ValueError(
            f"Non-finite {name}: {value!r}"
        )

    return x


def mean(values):
    return sum(values) / len(values)


def quantile(values, q):
    """
    Linear interpolation quantile,
    comparable to common dataframe quantile behavior.
    """
    xs = sorted(values)

    if not xs:
        return None

    if len(xs) == 1:
        return xs[0]

    pos = (len(xs) - 1) * q
    lo = math.floor(pos)
    hi = math.ceil(pos)

    if lo == hi:
        return xs[lo]

    frac = pos - lo

    return (
        xs[lo] * (1 - frac)
        + xs[hi] * frac
    )


def average_percentile_ranks(values):
    """
    Average ranks for ties.
    Output range: 1/n ... 1.
    """
    n = len(values)

    order = sorted(
        range(n),
        key=lambda i: values[i]
    )

    ranks = [None] * n
    pos = 0

    while pos < n:

        start = pos
        value = values[order[pos]]

        while (
            pos + 1 < n
            and values[order[pos + 1]] == value
        ):
            pos += 1

        end = pos

        # 1-based average rank
        avg_rank = (
            (start + 1) + (end + 1)
        ) / 2.0

        percentile = avg_rank / n

        for k in range(start, end + 1):
            ranks[order[k]] = percentile

        pos += 1

    return ranks


def pearson_corr(x, y):

    if len(x) != len(y):
        raise ValueError("Correlation lengths differ.")

    n = len(x)

    if n < 2:
        return None

    mx = mean(x)
    my = mean(y)

    num = sum(
        (a - mx) * (b - my)
        for a, b in zip(x, y)
    )

    den_x = sum(
        (a - mx) ** 2
        for a in x
    )

    den_y = sum(
        (b - my) ** 2
        for b in y
    )

    den = math.sqrt(
        den_x * den_y
    )

    if den == 0:
        return None

    return num / den


# ------------------------------------------------------------
# 1. PREFLIGHT
# ------------------------------------------------------------

for path in [
    PALMSIM_BASE,
    REP_SAMPLE,
    APSIM_FROZEN,
]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Files found: PASS")


# ------------------------------------------------------------
# 2. LOAD REPRESENTATIVE SAMPLE
# ------------------------------------------------------------

sample_rows = read_csv_dict(REP_SAMPLE)

sample_ids = [
    str(r["UNIT_ID"])
    for r in sample_rows
]

sample_set = set(sample_ids)

print("Representative sample :", len(sample_rows))

if (
    len(sample_rows) != 48
    or len(sample_set) != 48
):
    raise ValueError(
        "Representative sample must contain "
        "48 unique UNIT_ID."
    )


# ------------------------------------------------------------
# 3. PALMSIM BASELINE
# ------------------------------------------------------------

palm_rows = read_csv_dict(PALMSIM_BASE)

if not palm_rows:
    raise ValueError("PALMSIM baseline is empty.")

palm_columns = list(
    palm_rows[0].keys()
)

preferred = [
    "TRAILING_1Y_MEAN_T_HA_YR",
    "POTENTIAL_T_HA_YR",
    "BASELINE_T_HA_YR",
    "FFB_T_HA_YR",
]

palm_yield_col = None

for col in preferred:
    if col in palm_columns:
        palm_yield_col = col
        break

if palm_yield_col is None:

    candidates = []

    for col in palm_columns:
        cu = col.upper()

        if (
            "T_HA_YR" in cu
            and (
                "TRAIL" in cu
                or "POTENTIAL" in cu
                or "BASE" in cu
            )
        ):
            candidates.append(col)

    if len(candidates) != 1:
        raise ValueError(
            "Could not uniquely identify "
            f"PALMSIM yield column: {candidates}"
        )

    palm_yield_col = candidates[0]

print(
    "PALMSIM yield field    :",
    palm_yield_col
)

palm_map = {}

for row in palm_rows:

    uid = str(row["UNIT_ID"])

    if uid not in sample_set:
        continue

    if uid in palm_map:
        raise ValueError(
            f"Duplicate PALMSIM UNIT_ID: {uid}"
        )

    palm_map[uid] = to_float(
        row[palm_yield_col],
        f"PALMSIM {uid}"
    )

if len(palm_map) != 48:
    raise ValueError(
        f"Expected 48 PALMSIM rows; "
        f"found {len(palm_map)}"
    )

print("PALMSIM matched        :", len(palm_map))


# ------------------------------------------------------------
# 4. APSIM FROZEN
# ------------------------------------------------------------

apsim_rows = read_csv_dict(
    APSIM_FROZEN
)

if len(apsim_rows) != 48:
    raise ValueError(
        f"Expected 48 APSIM rows; "
        f"found {len(apsim_rows)}"
    )

apsim_map = {}

for row in apsim_rows:

    uid = str(row["UNIT_ID"])

    if uid in apsim_map:
        raise ValueError(
            f"Duplicate APSIM UNIT_ID: {uid}"
        )

    apsim_map[uid] = row

if set(apsim_map) != sample_set:
    raise ValueError(
        "APSIM ASU set differs from "
        "representative sample."
    )

print("APSIM matched          :", len(apsim_map))


# ------------------------------------------------------------
# 5. BUILD COMPARISON
# ------------------------------------------------------------

comparison = []

A = []
P = []

for uid in sample_ids:

    ar = apsim_map[uid]

    apsim_yield = to_float(
        ar["APSIM_A143_BASE_T_FFB_HA_YR"],
        f"APSIM {uid}"
    )

    palmsim_yield = palm_map[uid]

    delta = apsim_yield - palmsim_yield
    abs_diff = abs(delta)

    denom = (
        abs(apsim_yield)
        + abs(palmsim_yield)
    )

    if denom > 0:
        symmetric_diff = (
            200.0
            * abs_diff
            / denom
        )
    else:
        symmetric_diff = 0.0

    # SAFE ratio:
    # no division is executed at all
    # when PALMSIM <= 0.1.
    if palmsim_yield > 0.1:
        ratio = (
            apsim_yield
            / palmsim_yield
        )
    else:
        ratio = None

    record = {
        "UNIT_ID": uid,
        "AGE_PROXY_YR":
            ar["AGE_PROXY_YR"],
        "CLIMATE_PAIR_ID":
            ar["CLIMATE_PAIR_ID"],
        "PALMSIM_BASE_T_FFB_HA_YR":
            palmsim_yield,
        "APSIM_A143_BASE_T_FFB_HA_YR":
            apsim_yield,
        "MODEL_DELTA_APSIM_MINUS_PALMSIM_T_HA_YR":
            delta,
        "MODEL_ABS_DIFF_T_HA_YR":
            abs_diff,
        "MODEL_SYMMETRIC_DIFF_PCT":
            symmetric_diff,
        "APSIM_TO_PALMSIM_RATIO":
            ratio,
    }

    comparison.append(record)

    P.append(palmsim_yield)
    A.append(apsim_yield)


# ------------------------------------------------------------
# 6. RANK COMPARISON
# ------------------------------------------------------------

p_rank = average_percentile_ranks(P)
a_rank = average_percentile_ranks(A)

rank_diffs = []

for i, row in enumerate(comparison):

    row[
        "PALMSIM_PERCENTILE_RANK"
    ] = p_rank[i]

    row[
        "APSIM_PERCENTILE_RANK"
    ] = a_rank[i]

    rd = abs(
        a_rank[i]
        - p_rank[i]
    )

    row[
        "MODEL_RANK_ABS_DIFF"
    ] = rd

    rank_diffs.append(rd)


# ------------------------------------------------------------
# 7. GLOBAL METRICS
# ------------------------------------------------------------

deltas = [
    a - p
    for a, p in zip(A, P)
]

abs_diffs = [
    abs(d)
    for d in deltas
]

mae = mean(abs_diffs)

rmse = math.sqrt(
    mean([
        d ** 2
        for d in deltas
    ])
)

bias_mean = mean(deltas)
bias_median = statistics.median(deltas)

pearson_r = pearson_corr(
    P,
    A
)

spearman_rho = pearson_corr(
    p_rank,
    a_rank
)

symdiffs = [
    r["MODEL_SYMMETRIC_DIFF_PCT"]
    for r in comparison
]

ratios = [
    r["APSIM_TO_PALMSIM_RATIO"]
    for r in comparison
    if r["APSIM_TO_PALMSIM_RATIO"]
    is not None
]


# ------------------------------------------------------------
# 8. QC
# ------------------------------------------------------------

climate_pairs = set(
    str(r["CLIMATE_PAIR_ID"])
    for r in comparison
)

checks = {
    "comparison_rows_48":
        len(comparison) == 48,

    "unique_asu_48":
        len({
            r["UNIT_ID"]
            for r in comparison
        }) == 48,

    "same_representative_set":
        {
            r["UNIT_ID"]
            for r in comparison
        } == sample_set,

    "climate_pairs_12":
        len(climate_pairs) == 12,

    "all_palmsim_finite":
        all(
            math.isfinite(x)
            for x in P
        ),

    "all_apsim_finite":
        all(
            math.isfinite(x)
            for x in A
        ),

    "all_abs_diff_finite":
        all(
            math.isfinite(x)
            for x in abs_diffs
        ),
}

status = (
    "PASS_COMPARISON_COMPLETE"
    if all(checks.values())
    else "REVIEW"
)


# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

summary = {
    "palmsim_min_t_ha_yr":
        min(P),

    "palmsim_median_t_ha_yr":
        statistics.median(P),

    "palmsim_mean_t_ha_yr":
        mean(P),

    "palmsim_max_t_ha_yr":
        max(P),

    "apsim_min_t_ha_yr":
        min(A),

    "apsim_median_t_ha_yr":
        statistics.median(A),

    "apsim_mean_t_ha_yr":
        mean(A),

    "apsim_max_t_ha_yr":
        max(A),

    "mean_bias_apsim_minus_palmsim_t_ha_yr":
        bias_mean,

    "median_bias_apsim_minus_palmsim_t_ha_yr":
        bias_median,

    "mae_t_ha_yr":
        mae,

    "rmse_t_ha_yr":
        rmse,

    "pearson_r":
        pearson_r,

    "spearman_rho":
        spearman_rho,

    "median_symmetric_difference_pct":
        statistics.median(symdiffs),

    "p95_symmetric_difference_pct":
        quantile(symdiffs, 0.95),

    "median_absolute_rank_difference":
        statistics.median(rank_diffs),

    "p95_absolute_rank_difference":
        quantile(rank_diffs, 0.95),

    "median_apsim_to_palmsim_ratio":
        (
            statistics.median(ratios)
            if ratios
            else None
        ),

    "palmsim_zero_or_near_zero_count":
        sum(
            1
            for x in P
            if x <= 0.1
        ),
}


# ------------------------------------------------------------
# 10. SAVE CSV
# ------------------------------------------------------------

for row in comparison:

    row[
        "COMPARISON_STATUS"
    ] = "PROCESS_MODEL_DIAGNOSTIC_ONLY"

    row[
        "LOCAL_YIELD_VALIDATION"
    ] = False

    row[
        "MODEL_AGGREGATION_APPLIED"
    ] = False

    row[
        "COMPARISON_VERSION"
    ] = "40E01A_V3"


fieldnames = list(
    comparison[0].keys()
)

with open(
    OUT_CSV,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(comparison)


# ------------------------------------------------------------
# 11. SAVE QC JSON
# ------------------------------------------------------------

qc = {
    "gate":
        "40E01A_V3_PALMSIM_APSIM_CORE_COMPARISON",

    "status":
        status,

    "implementation":
        "PURE_PYTHON_STANDARD_LIBRARY",

    "representative_asus":
        len(comparison),

    "climate_pairs":
        len(climate_pairs),

    "palmsim_yield_field":
        palm_yield_col,

    "apsim_yield_field":
        "APSIM_A143_BASE_T_FFB_HA_YR",

    "unit":
        "t fresh FFB/ha/year",

    "summary":
        summary,

    "checks":
        checks,

    "interpretation_limits": [
        "Model-to-model comparison only.",
        "Neither model is ground truth.",
        "Not validation against observed plantation FFB.",
        "Bias is APSIM minus PALMSIM, not prediction error.",
        "Correlation describes co-variation/ranking, not accuracy.",
        "No PALMSIM/APSIM averaging is applied.",
    ],

    "next":
        (
            "40E01B discrepancy diagnostics "
            "by age and climate pair."
        ),
}

with open(
    OUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        qc,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 12. PRINT
# ------------------------------------------------------------

print("\n====================================================")
print("40E01A-V3 FINISHED")
print("====================================================")

print("Status            :", status)
print("ASUs              :", len(comparison))
print("Climate pairs     :", len(climate_pairs))

print("\nPALMSIM")
print(
    "Min               :",
    summary["palmsim_min_t_ha_yr"]
)
print(
    "Median            :",
    summary["palmsim_median_t_ha_yr"]
)
print(
    "Mean              :",
    summary["palmsim_mean_t_ha_yr"]
)
print(
    "Max               :",
    summary["palmsim_max_t_ha_yr"]
)

print("\nAPSIM")
print(
    "Min               :",
    summary["apsim_min_t_ha_yr"]
)
print(
    "Median            :",
    summary["apsim_median_t_ha_yr"]
)
print(
    "Mean              :",
    summary["apsim_mean_t_ha_yr"]
)
print(
    "Max               :",
    summary["apsim_max_t_ha_yr"]
)

print("\nMODEL-TO-MODEL")
print(
    "Mean bias A-P     :",
    bias_mean
)
print(
    "Median bias A-P   :",
    bias_median
)
print(
    "MAE               :",
    mae
)
print(
    "RMSE              :",
    rmse
)
print(
    "Pearson r         :",
    pearson_r
)
print(
    "Spearman rho      :",
    spearman_rho
)
print(
    "Median sym diff % :",
    summary[
        "median_symmetric_difference_pct"
    ]
)
print(
    "Median rank diff  :",
    summary[
        "median_absolute_rank_difference"
    ]
)
print(
    "PALMSIM <= 0.1    :",
    summary[
        "palmsim_zero_or_near_zero_count"
    ]
)

print("\nQC:")

for key, value in checks.items():
    print(
        f"{key:42s}: {value}"
    )

print("\nComparison CSV :", OUT_CSV)
print("QC JSON        :", OUT_JSON)

if status == "PASS_COMPARISON_COMPLETE":
    print(
        "Next           : 40E01B discrepancy diagnostics "
        "by age and climate pair."
    )
else:
    print(
        "Next           : Review comparison integrity."
    )

print("====================================================")

40E01A-V3 — PALMSIM vs APSIM CORE COMPARISON
PURE PYTHON / NO NUMPY / NO PANDAS / NO SCIPY
Files found: PASS
Representative sample : 48
PALMSIM yield field    : TRAILING_1Y_MEAN_T_HA_YR
PALMSIM matched        : 48
APSIM matched          : 48

40E01A-V3 FINISHED
Status            : PASS_COMPARISON_COMPLETE
ASUs              : 48
Climate pairs     : 12

PALMSIM
Min               : 0.0
Median            : 12.252312142273066
Mean              : 13.399248171155222
Max               : 26.693160047929048

APSIM
Min               : 0.1267306766622727
Median            : 19.297824782811592
Mean              : 17.435889214450146
Max               : 33.850461798272846

MODEL-TO-MODEL
Mean bias A-P     : 4.03664104329492
Median bias A-P   : 6.570358280365114
MAE               : 9.737694577749268
RMSE              : 11.729234983680062
Pearson r         : 0.06847234601481739
Spearman rho      : 0.11241077975209303
Median sym diff % : 78.82768186574874
Median rank diff  : 0.20833333333333334
PALMSIM 

In [3]:
# ============================================================
# OUTPUT 4 / 40E01B
# PALMSIM vs APSIM — DISCREPANCY DIAGNOSTICS
# BY AGE GROUP + CLIMATE PAIR
# SIAK_AOI01
#
# PURE PYTHON
# NO numpy / pandas / scipy
# ============================================================

from pathlib import Path
import csv
import json
import math
import statistics

WORKDIR = Path(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD"
)

IN_CSV = WORKDIR / (
    "40E01A_V3_PALMSIM_APSIM_CORE_COMPARISON_SIAK_AOI01.csv"
)

AGE_OUT = WORKDIR / (
    "40E01B_AGE_DISCREPANCY_SUMMARY_SIAK_AOI01_V1.csv"
)

CLIMATE_OUT = WORKDIR / (
    "40E01B_CLIMATE_PAIR_DISCREPANCY_SUMMARY_SIAK_AOI01_V1.csv"
)

TOP_OUT = WORKDIR / (
    "40E01B_TOP_MODEL_DISAGREEMENT_ASUS_SIAK_AOI01_V1.csv"
)

QC_OUT = WORKDIR / (
    "40E01B_DISCREPANCY_DIAGNOSTIC_QC_SIAK_AOI01_V1.json"
)

print("====================================================")
print("40E01B — PALMSIM vs APSIM DISCREPANCY DIAGNOSTICS")
print("PURE PYTHON")
print("====================================================")


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def read_csv_dict(path):

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        return list(
            csv.DictReader(f)
        )


def to_float(value, label="value"):

    try:
        x = float(value)

    except Exception:
        raise ValueError(
            f"Cannot convert {label}={value!r} to float."
        )

    if not math.isfinite(x):
        raise ValueError(
            f"Non-finite {label}: {value!r}"
        )

    return x


def mean(values):

    if not values:
        return None

    return sum(values) / len(values)


def quantile(values, q):

    if not values:
        return None

    xs = sorted(values)

    if len(xs) == 1:
        return xs[0]

    pos = (len(xs) - 1) * q

    lo = math.floor(pos)
    hi = math.ceil(pos)

    if lo == hi:
        return xs[lo]

    frac = pos - lo

    return (
        xs[lo] * (1 - frac)
        + xs[hi] * frac
    )


def average_ranks(values):

    n = len(values)

    order = sorted(
        range(n),
        key=lambda i: values[i]
    )

    ranks = [0.0] * n

    i = 0

    while i < n:

        j = i

        while (
            j + 1 < n
            and values[order[j + 1]]
            == values[order[i]]
        ):
            j += 1

        avg_rank = (
            (i + 1)
            + (j + 1)
        ) / 2.0

        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank

        i = j + 1

    return ranks


def correlation(x, y):

    if len(x) != len(y):
        raise ValueError(
            "Correlation arrays have different lengths."
        )

    if len(x) < 2:
        return None

    mx = mean(x)
    my = mean(y)

    dx = [
        v - mx
        for v in x
    ]

    dy = [
        v - my
        for v in y
    ]

    denom = math.sqrt(
        sum(v * v for v in dx)
        *
        sum(v * v for v in dy)
    )

    if denom == 0:
        return None

    return (
        sum(
            a * b
            for a, b in zip(dx, dy)
        )
        / denom
    )


def spearman(x, y):

    return correlation(
        average_ranks(x),
        average_ranks(y)
    )


def summarize_group(
    rows,
    group_field,
    group_value
):

    palmsim = [
        to_float(
            r["PALMSIM_BASE_T_FFB_HA_YR"]
        )
        for r in rows
    ]

    apsim = [
        to_float(
            r["APSIM_A143_BASE_T_FFB_HA_YR"]
        )
        for r in rows
    ]

    ages = [
        to_float(
            r["AGE_PROXY_YR"]
        )
        for r in rows
    ]

    delta = [
        a - p
        for a, p in zip(
            apsim,
            palmsim
        )
    ]

    abs_diff = [
        abs(x)
        for x in delta
    ]

    symmetric_diff = [
        to_float(
            r["MODEL_SYMMETRIC_DIFF_PCT"]
        )
        for r in rows
    ]

    rank_diff = [
        to_float(
            r["MODEL_RANK_ABS_DIFF"]
        )
        for r in rows
    ]

    return {

        group_field:
            group_value,

        "N_ASU":
            len(rows),

        "AGE_MIN":
            min(ages),

        "AGE_MEDIAN":
            statistics.median(ages),

        "AGE_MAX":
            max(ages),

        "PALMSIM_MEDIAN_T_HA_YR":
            statistics.median(palmsim),

        "APSIM_MEDIAN_T_HA_YR":
            statistics.median(apsim),

        "BIAS_MEAN_APSIM_MINUS_PALMSIM":
            mean(delta),

        "BIAS_MEDIAN_APSIM_MINUS_PALMSIM":
            statistics.median(delta),

        "MAE_T_HA_YR":
            mean(abs_diff),

        "RMSE_T_HA_YR":
            math.sqrt(
                mean([
                    x * x
                    for x in delta
                ])
            ),

        "MEDIAN_SYMMETRIC_DIFF_PCT":
            statistics.median(
                symmetric_diff
            ),

        "P95_SYMMETRIC_DIFF_PCT":
            quantile(
                symmetric_diff,
                0.95
            ),

        "MEDIAN_RANK_ABS_DIFF":
            statistics.median(
                rank_diff
            ),

        "PEARSON_R":
            correlation(
                palmsim,
                apsim
            ),

        "SPEARMAN_RHO":
            spearman(
                palmsim,
                apsim
            ),

        "APSIM_GT_PALMSIM_COUNT":
            sum(
                1
                for x in delta
                if x > 0
            ),

        "APSIM_LT_PALMSIM_COUNT":
            sum(
                1
                for x in delta
                if x < 0
            ),

        "PALMSIM_LE_0P1_COUNT":
            sum(
                1
                for x in palmsim
                if x <= 0.1
            ),
    }


def write_rows(
    path,
    rows
):

    if not rows:
        return

    with open(
        path,
        "w",
        encoding="utf-8",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=list(
                rows[0].keys()
            )
        )

        writer.writeheader()
        writer.writerows(rows)


# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

if not IN_CSV.exists():
    raise FileNotFoundError(
        IN_CSV
    )

rows = read_csv_dict(
    IN_CSV
)

print("Input rows :", len(rows))

if len(rows) != 48:
    raise ValueError(
        f"Expected 48 comparison rows; got {len(rows)}."
    )

if len({
    r["UNIT_ID"]
    for r in rows
}) != 48:

    raise ValueError(
        "Comparison UNIT_ID is not unique."
    )


# ------------------------------------------------------------
# 2. AGE-GROUP DIAGNOSTICS
# ------------------------------------------------------------

# Descriptive bins only.
# These are NOT agronomic management classes.

age_bins = [

    (
        "AGE_0_3",
        0,
        3
    ),

    (
        "AGE_4_7",
        4,
        7
    ),

    (
        "AGE_8_12",
        8,
        12
    ),

    (
        "AGE_13_18",
        13,
        18
    ),

    (
        "AGE_19_PLUS",
        19,
        999
    ),
]

age_summary = []

for label, lo, hi in age_bins:

    subset = [
        r
        for r in rows
        if (
            lo
            <= to_float(
                r["AGE_PROXY_YR"]
            )
            <= hi
        )
    ]

    if subset:

        age_summary.append(
            summarize_group(
                subset,
                "AGE_GROUP",
                label
            )
        )

write_rows(
    AGE_OUT,
    age_summary
)


# ------------------------------------------------------------
# 3. CLIMATE-PAIR DIAGNOSTICS
# ------------------------------------------------------------

climate_pairs = sorted(
    set(
        r["CLIMATE_PAIR_ID"]
        for r in rows
    )
)

climate_summary = []

for pair in climate_pairs:

    subset = [
        r
        for r in rows
        if (
            r["CLIMATE_PAIR_ID"]
            == pair
        )
    ]

    climate_summary.append(
        summarize_group(
            subset,
            "CLIMATE_PAIR_ID",
            pair
        )
    )

write_rows(
    CLIMATE_OUT,
    climate_summary
)


# ------------------------------------------------------------
# 4. TOP MODEL DISAGREEMENT ASUs
# ------------------------------------------------------------

sorted_rows = sorted(
    rows,
    key=lambda r:
        to_float(
            r[
                "MODEL_ABS_DIFF_T_HA_YR"
            ]
        ),
    reverse=True
)

top_rows = []

for rank, r in enumerate(
    sorted_rows[:12],
    start=1
):

    top_rows.append({

        "DISAGREEMENT_RANK":
            rank,

        "UNIT_ID":
            r["UNIT_ID"],

        "AGE_PROXY_YR":
            r["AGE_PROXY_YR"],

        "CLIMATE_PAIR_ID":
            r["CLIMATE_PAIR_ID"],

        "PALMSIM_T_FFB_HA_YR":
            r[
                "PALMSIM_BASE_T_FFB_HA_YR"
            ],

        "APSIM_T_FFB_HA_YR":
            r[
                "APSIM_A143_BASE_T_FFB_HA_YR"
            ],

        "DELTA_APSIM_MINUS_PALMSIM":
            r[
                "MODEL_DELTA_APSIM_MINUS_PALMSIM_T_HA_YR"
            ],

        "ABS_DIFF_T_HA_YR":
            r[
                "MODEL_ABS_DIFF_T_HA_YR"
            ],

        "SYMMETRIC_DIFF_PCT":
            r[
                "MODEL_SYMMETRIC_DIFF_PCT"
            ],

        "RANK_ABS_DIFF":
            r[
                "MODEL_RANK_ABS_DIFF"
            ],
    })

write_rows(
    TOP_OUT,
    top_rows
)


# ------------------------------------------------------------
# 5. IDENTIFY DIAGNOSTIC HOTSPOTS
# ------------------------------------------------------------

worst_age = max(
    age_summary,
    key=lambda r:
        r["MAE_T_HA_YR"]
)

worst_climate = max(
    climate_summary,
    key=lambda r:
        r["MAE_T_HA_YR"]
)


# ------------------------------------------------------------
# 6. QC
# ------------------------------------------------------------

checks = {

    "input_rows_48":
        len(rows) == 48,

    "unique_asu_48":
        len({
            r["UNIT_ID"]
            for r in rows
        }) == 48,

    "age_groups_created":
        len(age_summary) >= 1,

    "climate_pairs_12":
        len(
            climate_summary
        ) == 12,

    "four_asus_per_climate_pair":
        all(
            r["N_ASU"] == 4
            for r in climate_summary
        ),

    "top_disagreement_rows_12":
        len(top_rows) == 12,
}

status = (
    "PASS_DIAGNOSTICS_COMPLETE"
    if all(
        checks.values()
    )
    else "REVIEW"
)


# ------------------------------------------------------------
# 7. SAVE QC
# ------------------------------------------------------------

qc = {

    "gate":
        "40E01B_PALMSIM_APSIM_DISCREPANCY_DIAGNOSTICS",

    "status":
        status,

    "representative_asus":
        48,

    "age_groups":
        len(age_summary),

    "climate_pairs":
        len(climate_summary),

    "worst_age_group_by_mae":
        worst_age[
            "AGE_GROUP"
        ],

    "worst_age_group_mae_t_ha_yr":
        worst_age[
            "MAE_T_HA_YR"
        ],

    "worst_climate_pair_by_mae":
        worst_climate[
            "CLIMATE_PAIR_ID"
        ],

    "worst_climate_pair_mae_t_ha_yr":
        worst_climate[
            "MAE_T_HA_YR"
        ],

    "checks":
        checks,

    "interpretation_limits": [

        "Age bins are descriptive only.",

        "Climate-pair summaries contain only four representative ASUs each.",

        "Large model differences are structural/scenario disagreement, not observed prediction error.",

        "Neither model is treated as ground truth.",

        "No APSIM/PALMSIM averaging is applied.",
    ],

    "next":
        (
            "40E02 model-agreement synthesis "
            "and benchmark-role decision."
        ),
}

with open(
    QC_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        qc,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 8. PRINT
# ------------------------------------------------------------

print("\n====================================================")
print("40E01B FINISHED")
print("====================================================")

print(
    "Status                 :",
    status
)

print(
    "Age groups             :",
    len(age_summary)
)

print(
    "Climate pairs          :",
    len(climate_summary)
)

print(
    "\nWorst age group by MAE :",
    worst_age[
        "AGE_GROUP"
    ]
)

print(
    "Age-group MAE          :",
    worst_age[
        "MAE_T_HA_YR"
    ],
    "t FFB/ha/yr"
)

print(
    "\nWorst climate pair     :",
    worst_climate[
        "CLIMATE_PAIR_ID"
    ]
)

print(
    "Climate-pair MAE       :",
    worst_climate[
        "MAE_T_HA_YR"
    ],
    "t FFB/ha/yr"
)

print("\nAGE SUMMARY")

for r in age_summary:

    print(
        r["AGE_GROUP"],
        "| N =", r["N_ASU"],
        "| P median =", round(
            r["PALMSIM_MEDIAN_T_HA_YR"],
            3
        ),
        "| A median =", round(
            r["APSIM_MEDIAN_T_HA_YR"],
            3
        ),
        "| MAE =", round(
            r["MAE_T_HA_YR"],
            3
        ),
        "| rho =", (
            round(
                r["SPEARMAN_RHO"],
                3
            )
            if r["SPEARMAN_RHO"]
            is not None
            else None
        )
    )

print("\nQC:")

for k, v in checks.items():

    print(
        f"{k:42s}: {v}"
    )

print("\nAge summary     :", AGE_OUT)
print("Climate summary :", CLIMATE_OUT)
print("Top disagreement:", TOP_OUT)
print("QC JSON         :", QC_OUT)

if status == "PASS_DIAGNOSTICS_COMPLETE":

    print(
        "Next            : 40E02 model-agreement "
        "synthesis and benchmark-role decision."
    )

else:

    print(
        "Next            : Review diagnostics before 40E02."
    )

print("====================================================")

40E01B — PALMSIM vs APSIM DISCREPANCY DIAGNOSTICS
PURE PYTHON
Input rows : 48

40E01B FINISHED
Status                 : PASS_DIAGNOSTICS_COMPLETE
Age groups             : 5
Climate pairs          : 12

Worst age group by MAE : AGE_4_7
Age-group MAE          : 19.418899028859233 t FFB/ha/yr

Worst climate pair     : CHIRPS_R985_C5634__ERA5L_+00.7_+101.7
Climate-pair MAE       : 13.657203294786033 t FFB/ha/yr

AGE SUMMARY
AGE_0_3 | N = 4 | P median = 1.398 | A median = 0.436 | MAE = 1.091 | rho = 0.738
AGE_4_7 | N = 6 | P median = 23.607 | A median = 3.823 | MAE = 19.419 | rho = -0.714
AGE_8_12 | N = 11 | P median = 15.264 | A median = 21.142 | MAE = 7.593 | rho = 0.682
AGE_13_18 | N = 9 | P median = 17.351 | A median = 19.085 | MAE = 5.861 | rho = -0.25
AGE_19_PLUS | N = 18 | P median = 7.401 | A median = 19.402 | MAE = 11.681 | rho = 0.261

QC:
input_rows_48                             : True
unique_asu_48                             : True
age_groups_created                        : T

In [4]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40E02_V2_FINAL_MODEL_ROLE_WALL_TO_WALL_COVERAGE_CONTRACT_SIAK_AOI01.py",
    encoding="utf-8"
).read())

40E02-V2 — FINAL MODEL ROLE + WALL-TO-WALL CONTRACT
PURE PYTHON / NO MODEL RERUN
Input files found              : PASS
Master ASUs                    : 820
PALMSIM direct ASUs            : 654
Fallback-path ASUs             : 166
PALMSIM yield field            : TRAILING_1Y_MEAN_T_HA_YR
APSIM benchmark ASUs           : 48
Pearson r                      : 0.06847234601481739
Spearman rho                   : 0.11241077975209303

40E02-V2 FINISHED
Status                         : PASS_WALL_TO_WALL_ROLE_FREEZE

MODEL ROLE
PALMSIM                        : PRIMARY_PROCESS_MODEL_FOR_NUMERIC_PRODUCTIVITY_POTENTIAL
APSIM                          : REPRESENTATIVE_PROCESS_SCENARIO_BENCHMARK_ONLY
Model averaging                : NO
Cross-model bias correction    : NO

WALL-TO-WALL COVERAGE
Master ASUs                    : 820
Direct PALMSIM pathway         : 654
Fallback pathway               : 166
Spatially included             : 820
Final map target               : 820 / 820 ASUs

MODEL COMPARIS

In [5]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40E03A_FALLBACK_PRODUCTIVITY_FEATURE_AVAILABILITY_AUDIT_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40E03A — FALLBACK FEATURE AVAILABILITY & DESIGN AUDIT
PURE PYTHON / NO MODEL RERUN
Master ASUs                    : 820
Direct donors                  : 654
Fallback recipients            : 166

40E03A FINISHED
Status                         : PASS_FEATURE_AUDIT_READY_FOR_40E03B
Candidate CSV files            : 31
Condition candidate files      : 5
Soil candidate files           : 7
Age-evidence candidate files   : 9
Climate candidate files        : 5

TOP CONDITION CANDIDATES
- 40B05_COMMON_MODEL_INPUT_MASTER_SIAK_AOI01_V1.csv | overlap= 820 | cols= CONDITION_SCORE | CONDITION_VALID_FRACTION | CONDITION_AVAILABLE
- 40B03C1_FINAL_ASU_500M_SOIL_TEXTURE_SIAK_AOI01_V1.csv | overlap= 820 | cols= CONDITION_DATA_VALID | COND_P10 | COND_P50 | COND_P90 | CONDITION_RANGE_P90_P10 | CONDITION_VALID_FRACTION
- 40B04A_ASU_CLIMATE_CELL_MAPPING_SIAK_AOI01_V1.csv | overlap= 820 | cols= CONDITION_DATA_VALID | COND_P10 | COND_P50 | COND_P90 | CONDITION_RANGE_P90_P10 | CONDITION_VALID_FRACTION
- 40B02A_G

In [6]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40E03B_PEER_BASED_INDICATIVE_PRODUCTIVITY_RANGE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40E03B — PEER-BASED INDICATIVE PRODUCTIVITY RANGE
+ INTERNAL LEAVE-ONE-OUT RECONSTRUCTION
PURE PYTHON / NO MODEL RERUN
Direct donors                  : 654
Fallback recipients            : 166
PALMSIM yield field            : TRAILING_1Y_MEAN_T_HA_YR

40E03B FINISHED
Status                         : PASS_EXECUTION_DIAGNOSTIC_COMPLETE

FALLBACK 166
Rows                           : 166
Exact climate-pair match       : 157
Condition + soil               : 164
Soil only                      : 2
LOW confidence                 : 121
VERY_LOW confidence            : 45
Indicative P50 min/median/max  : 3.858605627913371 / 10.323350548261768 / 21.79088326007144
Median P10-P90 width           : 15.584367343263345 t FFB/ha/yr

INTERNAL LOO RECONSTRUCTION — 654
MAE                            : 5.5089129854996095
Median absolute error          : 3.4867735973829674
RMSE                           : 7.819938746652948
Mean bias recon-direct         : -0.5691741174415982
Pearson r                      :

In [7]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40E03C_FALLBACK_PRODUCTIVITY_INTERVAL_CALIBRATION_FREEZE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40E03C — FALLBACK INTERVAL CALIBRATION + FREEZE
PURE PYTHON / NO MODEL RERUN
Fallback rows                  : 166
LOO diagnostic rows            : 654
Raw P10-P90 containment       : 77.37003058103976 %
Q95 miss-distance expansion   : 3.497231378860263 t FFB/ha/yr

40E03C FINISHED
Status                         : PASS_FALLBACK_METHOD_FROZEN

40E03B INTERPRETATION
Raw LOO P10-P90 containment   : 77.37003058103976 %
Q95 miss-distance expansion   : 3.497231378860263 t FFB/ha/yr
Calibrated LOO containment    : 94.95412844036697 %

CALIBRATED LOO BY AGE
AGE_0_3 | N = 11 | containment = 45.45 % | median width = 23.975
AGE_4_7 | N = 67 | containment = 91.04 % | median width = 22.426
AGE_8_12 | N = 152 | containment = 98.03 % | median width = 22.385
AGE_13_18 | N = 170 | containment = 94.71 % | median width = 23.295
AGE_19_PLUS | N = 254 | containment = 96.46 % | median width = 22.873

FROZEN FALLBACK 166
Rows                           : 166
Center min/median/max          : 3.858605627913371 /

In [8]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40F01_WALL_TO_WALL_PRODUCTIVITY_POTENTIAL_SYNTHESIS_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40F01 — WALL-TO-WALL PRODUCTIVITY POTENTIAL SYNTHESIS
PURE PYTHON / NO MODEL RERUN
Master spatial domain           : 820
Direct PALMSIM estimates        : 654
PALMSIM yield field             : TRAILING_1Y_MEAN_T_HA_YR
Frozen fallback estimates       : 166

40F01 FINISHED
Status                         : PASS_WALL_TO_WALL_PRODUCTIVITY_SYNTHESIS

WALL-TO-WALL COVERAGE
Total ASUs                     : 820
Direct PALMSIM                 : 654
Indicative fallback            : 166
Productivity center coverage   : 820 / 820

DIRECT PALMSIM 654
Min / median / mean / max      : 0.0 / 15.15169994057731 / 14.289290830141967 / 26.69316409307493

FALLBACK 166
Center min/median/mean/max     : 3.858605627913371 / 10.323350548261768 / 11.494550899028098 / 21.79088326007144
Median final range width       : 22.578830100983872 t FFB/ha/yr
LOW / VERY_LOW confidence      : 121 / 45

ALL 820 CENTERS
Min / median / mean / max      : 0.0 / 12.273353581068132 / 13.723526405062827 / 26.69316409307493

QC:
wall_

In [9]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40G01_PRODUCTIVITY_CALIBRATION_THRESHOLD_SENSITIVITY_CONFIDENCE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40G01 — PRODUCTIVITY CALIBRATION + CONFIDENCE DESIGN
PURE PYTHON / NO MODEL RERUN
Master ASUs                    : 820
Direct reference ASUs          : 654
Fallback ASUs                  : 166

DIRECT PALMSIM REFERENCE
Min                            : 0.0
Q20                            : 6.261789816424667
Q25                            : 7.441314893073895
Q30                            : 7.859202617259128
Median                         : 15.15169994057731
Q70                            : 20.05705454758828
Q75                            : 21.635506413899424
Q80                            : 22.093848381916104
Max                            : 26.69316409307493

STRUCTURAL DISAGREEMENT REFERENCE
Q33                            : 10.57197406085994
Q67                            : 11.681498484878192

40G01 FINISHED
Status                         : PASS_CALIBRATION_DIAGNOSTIC_COMPLETE

BASELINE PRODUCTIVITY BAND CANDIDATE
Reference                      : Direct PALMSIM 654
LOW <= Q25          

In [10]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40G02_PRODUCTIVITY_THRESHOLD_CONFIDENCE_FREEZE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40G02 — PRODUCTIVITY THRESHOLD + CONFIDENCE FREEZE
PURE PYTHON / NO MODEL RERUN
Master ASUs                    : 820
Direct PALMSIM                 : 654
Fallback                       : 166
Frozen LOW threshold <=        : 7.441314893073895
Frozen HIGH threshold >=       : 21.635506413899424

40G02 FINISHED
Status                         : PASS_PRODUCTIVITY_CALIBRATION_FROZEN

FROZEN PRODUCTIVITY THRESHOLDS
LOW <=                         : 7.441314893073895
HIGH >=                        : 21.635506413899424
Schema                         : Q25/Q75 direct PALMSIM
Interpretation                 : RELATIVE PILOT

SENSITIVITY
Q20/Q80 switches              : 10.731707317073171 %
Q30/Q70 switches              : 9.390243902439025 %
Max alternative switch        : 10.731707317073171 %

FINAL PRODUCTIVITY BAND COUNTS
{'RELATIVE_LOW': 199, 'RELATIVE_TYPICAL': 456, 'RELATIVE_HIGH': 165}

PRODUCTIVITY-BAND CONFIDENCE 820
{'MODERATE': 171, 'LOW': 420, 'VERY_LOW': 229}

DIRECT BAND CONFIDENCE 654


In [11]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40H01_MANAGEMENT_PRIORITY_INTEGRATION_RULE_SENSITIVITY_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40H01 — MANAGEMENT PRIORITY INTEGRATION + SENSITIVITY
PURE PYTHON / NO MODEL RERUN
Master ASUs                    : 820
Direct pathway                 : 654
Fallback pathway               : 166

CONDITION SCORE REFERENCE
Q40                           : 0.0006139571712933298
Q50                           : 0.15378866414523357
Q60                           : 1.8119206874069274
Q70                           : 4.357419242111058
Q75                           : 6.930309375194936
Q80                           : 9.434522506665562
Q85                           : 10.946066438090744
Q90                           : 12.89441926771219
Q95                           : 15.226911311142437

40H01 FINISHED
Status                         : PASS_MANAGEMENT_CANDIDATE_DIAGNOSTIC_COMPLETE

BASELINE CONDITION THRESHOLDS
WATCH >= Q50                  : 0.15378866414523357
ELEVATED >= Q75               : 6.930309375194936
HIGH >= Q90                   : 12.89441926771219
Class 4 age minimum           : 19 years p

In [12]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40H02_FINAL_MANAGEMENT_PRIORITY_FREEZE_STABILITY_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40H02 — FINAL MANAGEMENT PRIORITY FREEZE
+ PER-ASU RULE STABILITY + CONFIDENCE
PURE PYTHON / NO MODEL RERUN
Master ASUs                    : 820
Direct pathway                 : 654
Fallback pathway               : 166
Max alternative switch rate    : 8.902439024390244 %
Freeze decision                 : BASELINE ACCEPTED

40H02 FINISHED
Status                         : PASS_MANAGEMENT_PRIORITY_FROZEN

FINAL CLASS COUNTS — 820
Class 0 Insufficient Data / Limited Evidence  : 4
Class 1 Maintain                              : 159
Class 2 Monitor                               : 551
Class 3 Management Intervention Priority      : 103
Class 4 Replanting Evaluation Priority        : 3

DIRECT 654
{0: 0, 1: 159, 2: 430, 3: 62, 4: 3}

FALLBACK 166
{0: 4, 1: 0, 2: 121, 3: 41, 4: 0}

RULE STABILITY
{'STABLE_3_OF_3': 686, 'MODERATE_2_OF_3': 134, 'UNSTABLE_1_OF_3': 0}

MANAGEMENT CONFIDENCE
{'MODERATE': 116, 'LOW': 579, 'VERY_LOW': 125}

CLASS-SPECIFIC STABILITY
Class 0: {'STABLE_3_OF_3': 4, 'MODER

In [13]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I01_FINAL_SPATIAL_DOMAIN_QA_DELIVERY_FREEZE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40I01 — FINAL SPATIAL/DOMAIN QA + DELIVERY FREEZE
PURE PYTHON / NO MODEL RERUN
Final management rows           : 820
Exact spatial-domain rows       : 820

AREA FIELD
Selected field                  : ASU_AREA_HA_EXACT
Exact plantation area           : 13867.099001884668 ha
Difference vs frozen total      : 1.1533302313182503e-07 ha

WEATHER INPUT DISCOVERY
CHIRPS daily                    : 40B04B1_CHIRPS_V2_DAILY_NATIVE_CELLS_2001_2025_SIAK_AOI01_V1.csv
ERA5-Land daily                 : 40B04B2_ERA5LAND_DAILY_NATIVE_CELLS_2001_2025_SIAK_AOI01_V4.csv
CHIRPS cells aggregated         : 12
ERA5-Land cells aggregated      : 6

40I01 FINISHED
Status                         : PASS_FINAL_DELIVERY_FREEZE

SPATIAL DOMAIN
ASUs                           : 820
Exact plantation area          : 13867.099001884668 ha
Area field                     : ASU_AREA_HA_EXACT
Class-area total               : 13867.099001884691 ha

FINAL CLASS COUNT + AREA
Class 0 | N = 4 | area = 0.036 ha | 0.0 %
Class 1 | N 

In [ ]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_V4_LITE_MANAGEMENT_PRIORITY_WITH_PRODUCTIVITY_SIAK_AOI01.py",
    encoding="utf-8"
).read())

40I02A V4 — MANAGEMENT PRIORITY VISUAL PREVIEW
+ PRODUCTIVITY POTENTIAL TABLE
CSV ONLY / MEMORY SAFE
Frozen ASUs                    : 820
40B04A mapping rows            : 820

40B04A HEADER
['UNIT_ID', 'GRID_INDEX', 'GRID_SIZE_M', 'CELL_AREA_HA', 'ASU_AREA_HA', 'PLANTATION_FRACTION', 'AGE_DATA_VALID', 'AGE_P10', 'AGE_P50', 'AGE_P90', 'AGE_RANGE_P90_P10', 'AGE_VALID_FRACTION', 'CONDITION_DATA_VALID', 'COND_P10', 'COND_P50', 'COND_P90', 'CONDITION_RANGE_P90_P10', 'CONDITION_VALID_FRACTION', 'REFERENCE_YEAR', 'ASU_STAGE', 'ASU_AREA_HA_EXACT', 'PLANTATION_FRACTION_EXACT', 'CENTROID_LON_EXACT', 'CENTROID_LAT_EXACT', 'ASU_TYPE_FINAL', 'ASU_VERSION_FINAL', 'GEOMETRY_STATUS', 'SAND_PCT_0_100_MEAN', 'SAND_PCT_0_100_STD', 'SAND_PCT_0_100_COUNT', 'SILT_PCT_0_100_MEAN', 'SILT_PCT_0_100_STD', 'SILT_PCT_0_100_COUNT', 'CLAY_PCT_0_100_MEAN', 'CLAY_PCT_0_100_STD', 'CLAY_PCT_0_100_COUNT', 'TEXTURE_SUM_PCT', 'SOIL_TEXTURE_DATA_VALID', 'TEXTURE_SUM_QC', 'SOIL_SOURCE', 'SOIL_OBSERVED_LOCAL', 'SOIL_DEPTH_PR

In [1]:
print("kernel aman")

kernel aman


In [2]:
import matplotlib.pyplot as plt
print("matplotlib aman")

matplotlib aman


In [ ]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_V5_SIMPLE_VISUAL_MAP_TABLE_SIAK_AOI01.py",
    encoding="utf-8"
).read())

FINAL rows                     : 820
MAPPING rows                   : 820
Centroid                       : CENTROID_LON_EXACT / CENTROID_LAT_EXACT


In [1]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_V6_PURE_PYTHON_SVG_HTML_MAP_TABLE_SIAK_AOI01.py",
    encoding="utf-8"
).read())

FINAL rows                     : 820
MAPPING rows                   : 820
Renderer                       : pure Python SVG/HTML (no matplotlib)

40I02A V6 FINISHED

CLASS TABLE
Class 0 | N=4 | Area=0.04 ha | Potential=2.34–27.11 t FFB/ha/yr (Indicative range)
Class 1 | N=159 | Area=2932.53 ha | Potential=18.86 t FFB/ha/yr (Median potential)
Class 2 | N=551 | Area=8830.11 ha | Potential=13.34 t FFB/ha/yr (Median potential)
Class 3 | N=103 | Area=2040.44 ha | Potential=7.01 t FFB/ha/yr (Median potential)
Class 4 | N=3 | Area=63.98 ha | Potential=3.86 t FFB/ha/yr (Median potential)

HTML visualization             : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_SIMPLE_VISUAL_PREVIEW_SIAK_AOI01_V6\40I02A_MANAGEMENT_PRIORITY_MAP_TABLE_SIAK_AOI01_V6.html
SVG visualization              : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_SIMPLE_VISUAL_PREVIEW_SIAK_AOI01_V6\40I02A_MANAGEMENT_PRIORITY_MAP_TABLE_SIAK_AOI01_V6.svg
CSV table       

In [2]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_V7_EXACT_POLYGON_BROWSER_PREVIEW_SIAK_AOI01.py",
    encoding="utf-8"
).read())

40I02A V7 — EXACT POLYGON BROWSER PREVIEW
PURE PYTHON / NO MATPLOTLIB / NO GEOPANDAS
Frozen attributes              : 820
Loading exact GeoJSON          : 40B03A_FINAL_ASU_500M_EXACT_SIAK_AOI01_V1.geojson
Exact geometry features        : 820
UNIT_ID match                  : 820 / 820
Geometry types                 : ['MultiPolygon', 'Polygon']
Raw rings                      : 2481
Raw vertices                   : 32076
Class counts                   : {0: 4, 1: 159, 2: 551, 3: 103, 4: 3}
Building browser paths         : started
Browser paths                  : 820
Display rings                  : 2481
Display vertices               : 32076

40I02A V7 FINISHED
Status                         : PASS_40I02A_EXACT_POLYGON_BROWSER_PREVIEW
HTML preview                   : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_EXACT_POLYGON_BROWSER_PREVIEW_SIAK_AOI01_V1\40I02A_EXACT_ASU_MANAGEMENT_PRIORITY_PREVIEW_SIAK_AOI01_V1.html
QC JSON                        : C:\Users\

In [3]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_V8_REMOVE_ASU_BOUNDARY_DEFAULT_SIAK_AOI01.py",
    encoding="utf-8"
).read())

40I02A V8 FINISHED
Status                         : PASS_BOUNDARYLESS_DEFAULT
Input V7 HTML                  : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_EXACT_POLYGON_BROWSER_PREVIEW_SIAK_AOI01_V1\40I02A_EXACT_ASU_MANAGEMENT_PRIORITY_PREVIEW_SIAK_AOI01_V1.html
New HTML                       : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02A_EXACT_POLYGON_BROWSER_PREVIEW_SIAK_AOI01_V1\40I02A_EXACT_ASU_MANAGEMENT_PRIORITY_PREVIEW_NO_BOUNDARY_SIAK_AOI01_V2.html
Default ASU boundary           : OFF
Boundary checkbox              : AVAILABLE
Popup/hover                    : UNCHANGED
Scientific result changed      : False

Next                           : double-click NEW HTML.


In [4]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02B_WEB_DISPLAY_CONTRACT_FREEZE_SIAK_AOI01_V1.py",
    encoding="utf-8"
).read())

40I02B FINISHED
Status                         : PASS_40I02B_DISPLAY_CONTRACT_FROZEN
Class summary                  : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02B_WEB_DISPLAY_CONTRACT_SIAK_AOI01_V1\40I02B_CLASS_SUMMARY_WITH_PRODUCTIVITY_SIAK_AOI01_V1.csv
Legend contract                : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02B_WEB_DISPLAY_CONTRACT_SIAK_AOI01_V1\40I02B_LEGEND_SIAK_AOI01_V1.json
Popup contract                 : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02B_WEB_DISPLAY_CONTRACT_SIAK_AOI01_V1\40I02B_POPUP_FIELD_CONTRACT_SIAK_AOI01_V1.json
Display contract               : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40I02B_WEB_DISPLAY_CONTRACT_SIAK_AOI01_V1\40I02B_DISPLAY_CONTRACT_SIAK_AOI01_V1.json
40J01 reusable AOI config      : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40J01_WEB_PACKAGE_CONFIG_SIAK_AOI01_V1.json

CLASS PRODUCTIVITY SUM

In [5]:
exec(open(
    r"C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40J01_GENERIC_WEB_DEVELOPMENT_PACKAGE_BUILDER_V1.py",
    encoding="utf-8"
).read())

40J01 — GENERIC WEB PACKAGE BUILDER
AOI                            : SIAK_AOI01

40J01 FINISHED
Status                         : PASS_WEB_DEVELOPMENT_PACKAGE_READY
Package folder                 : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40J01_WEB_PACKAGE_SIAK_AOI01_v1
ZIP                            : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40J01_WEB_PACKAGE_SIAK_AOI01_v1.zip
Primary web GeoJSON            : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40J01_WEB_PACKAGE_SIAK_AOI01_v1\data\output4__siak_aoi01__management_priority__2025__v1.geojson
README                         : C:\Users\user\Documents\KERJA\SPATIAL DATA ANALYST INTERN\04_YIELD\40J01_WEB_PACKAGE_SIAK_AOI01_v1\README_WEB_DEVELOPMENT.txt
Scientific result changed      : False
